# 06_research_backtest.ipynb — SET50 Adaptive Fear & Alpha

Research Backtest แบบ fail-closed ใช้ข้อมูลใน repository เท่านั้น หากหลักฐานหรือ methodology สำคัญยังไม่พร้อม จะบันทึกเฉพาะ Audit/Blocker Report และไม่สร้างผล Backtest

## 1. Research overview และ frozen decisions

- Feature ณ วัน `t` ใช้สร้าง Prediction หลังตลาดปิด และคำสั่งจากวัน `t` อ้างอิง Raw Open ของ SET session ถัดไป `t+1` เท่านั้น
- Valuation ใช้ Raw Close ของ execution session; ไม่มีการ fill ด้วย Close วันก่อน, High/Low, ค่าเฉลี่ย OHLC หรือราคาที่ forward-fill
- ไม่มี Gap Limit; ราคา execution คือ `Raw Open × (1 ± slippage)` และคิด Fee จาก Fill Notional เพิ่มหนึ่งครั้ง
- Long-only, no leverage; Sell และ forced exit ก่อน Buy; Hold คงจำนวนหุ้นเดิม; Buy ใช้เงินสดที่มีจริงและจำกัดไม่เกิน 25% ของ deployable cash ต่อหุ้น
- Fee `0.16799%` ต่อขา และ slippage `0.10%` เป็นสมมติฐานงานวิจัย ไม่ใช่อัตราตลาดย้อนหลังที่ยืนยันแล้ว
- Sharpe ใช้ risk-free rate = 0 ต่อวันเป็นสมมติฐานที่เปิดเผย
- MPT objective ที่ตกลงไว้คือ maximize `w'μ - 5*w'Σw`, long-only, `sum(w) <= 1` และ `wi <= 0.25`; covariance policy ยังไม่อนุมัติ
- CVaR objective/constraint ยัง unresolved; ห้ามใช้วิธีที่ทำให้ all-cash เป็นคำตอบโดยปริยาย
- ห้ามดาวน์โหลดข้อมูล, แก้ Snapshot/Master/Model, refit/tune โมเดล, หรือใช้ Actual Target เป็น input ในทางตัดสินใจ
- CVaR formulation, board lot, stale-price policy, covariance/fallback และ Corporate Action accounting ที่ไม่มีหลักฐาน/ข้อตกลง จะคงเป็น blocker

## 2. Imports, environment และ reproducibility

In [14]:
# นำเข้าเฉพาะเครื่องมือที่ใช้ตรวจไฟล์ คำนวณ และทดสอบกลไก
import hashlib
import importlib.metadata
import json
import math
import platform
import sys
import uuid
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| scikit-learn:', sklearn.__version__)
#

Python: 3.11.12
Platform: Windows-10-10.0.26300-SP0
pandas: 3.1.1 | numpy: 2.4.6 | scikit-learn: 1.9.1


## 3. Project paths และ source inventory

หา project root จาก working directory/parent ของ Notebook; Execution Snapshot อ่านจาก lineage จริง

In [15]:
# ค้นหา root จากไฟล์ Notebook และโฟลเดอร์ข้อมูลที่มีอยู่จริง
def locate_project_root(start: Path | None = None) -> Path:
    origin = (start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        if (candidate / '06_research_backtest.ipynb').is_file() and (candidate / 'dataset').is_dir():
            return candidate
    raise FileNotFoundError('ไม่พบ project root ที่มี 06_research_backtest.ipynb และ dataset/')


PROJECT_ROOT = locate_project_root()
DATASET_DIR = PROJECT_ROOT / 'dataset'
BUILD_DIR = DATASET_DIR / 'verified_build_20261113_152948'
BACKTEST_DIR = PROJECT_ROOT / 'backtest_results'

FEATURE_MASTER_PATH = BUILD_DIR / 'SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet'
FEATURE_MASTER_FALLBACK_PATH = BUILD_DIR / 'SET50_Master_Dataset_DateSymbol_Sorted.parquet'
EXECUTION_SIDECAR_PATH = BUILD_DIR / 'SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.json'
LINEAGE_PATH = BACKTEST_DIR / 'execution_data_lineage.json'
VERIFIED_EXECUTION_STATUS_PATH = BACKTEST_DIR / 'verified_execution_data_status.json'
GATE_CHECKLIST_PATH = BACKTEST_DIR / 'research_gate_checklist.csv'
EXECUTION_AUDIT_PATH = BACKTEST_DIR / 'execution_price_integrity_audit.csv'
CORPORATE_ACTION_LEDGER_PATH = BACKTEST_DIR / 'corporate_action_ledger_official.csv'
CORPORATE_ACTION_VENDOR_PATH = BACKTEST_DIR / 'corporate_action_vendor_events.csv'
BANPU_AUDIT_PATH = BACKTEST_DIR / 'banpu_missing_data_audit.csv'
GLOBAL_TICK_AUDIT_PATH = BACKTEST_DIR / 'global_tick_basis_diagnostic.csv'
GLOBAL_RECENT_CROSSCHECK_PATH = BACKTEST_DIR / 'global_set_recent_crosscheck.csv'
SECURITY_IDENTITY_PATH = BACKTEST_DIR / 'security_identity_mapping.csv'
EXECUTION_TICK_AUDIT_PATH = BACKTEST_DIR / 'execution_price_tick_audit.csv'
EXECUTION_TICK_ORIGINAL_PATH = BACKTEST_DIR / 'execution_price_tick_audit_original.csv'
MEMBERSHIP_PATH = DATASET_DIR / 'SET50_constituents_by_halfyear_2016_2026.csv'
DECLARATION_PATH = DATASET_DIR / 'set50_declare_dates_2016_2026.csv'

FEATURE_COLUMNS = ['daily_return', 'momentum_20d', 'vol_z', 'VolRank', 'CorrRank', 'LiquidityRank']
TARGET_COLUMN = 'target'
TARGET_LABELS = {0: 'Panic', 1: 'FOMO', 2: 'Normal'}
EXPECTED_EXECUTION_SHA256 = '14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a'

# ค่าที่ผู้ใช้อนุมัติเป็น research assumptions
BUY_COST_RATE = 0.0016799
SELL_COST_RATE = 0.0016799
SLIPPAGE_RATE = 0.001
SHARPE_RISK_FREE_RATE_DAILY = 0.0
MAX_BUY_WEIGHT_PER_STOCK = 0.25
GAP_LIMIT = None
EXECUTION_LAG_SESSIONS = 1
INITIAL_EQUITY = 1_000_000.0

# ยังไม่มีหลักฐาน/อนุมัติพอ จึงไม่กำหนดค่าทดแทนเอง
BOARD_LOT_SIZE = None
MAX_STALE_CALENDAR_DAYS = None
COVARIANCE_ESTIMATOR = None
CVAR_FORMULATION = None
ALLOCATOR_FALLBACK_POLICY = None
ALLOCATION_LOOKBACK = {'EW': None, 'IV': 60, 'ERC': 60, 'HRP': 60, 'MinVar': 60, 'CVaR': 252, 'MPT': 60}
CVAR_CONFIDENCE = 0.95
MPT_RISK_AVERSION = 5.0

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Feature master exists:', FEATURE_MASTER_PATH.is_file() or FEATURE_MASTER_FALLBACK_PATH.is_file())
print('Execution sidecar exists:', EXECUTION_SIDECAR_PATH.is_file())
#

PROJECT_ROOT: D:\Project_Github\set_fear_index
Feature master exists: True
Execution sidecar exists: True


## 4. Snapshot / lineage verification และ read-only source loading

In [16]:
# Helper สำหรับ Hash และอ่านหลักฐานโดยไม่แก้ source
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def notebook_source_sha256(path: Path) -> str | None:
    if not path.is_file():
        return None
    payload = json.loads(path.read_text(encoding='utf-8'))
    for cell in payload.get('cells', []):
        if cell.get('cell_type') == 'code':
            cell['outputs'] = []
            cell['execution_count'] = None
    canonical = json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(canonical).hexdigest()


def read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    value = json.loads(path.read_text(encoding='utf-8'))
    return value if isinstance(value, dict) else {}


def read_csv_if_present(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()


def safe_project_path(value: str | None) -> Path | None:
    if not value:
        return None
    candidate = Path(value).expanduser().resolve()
    try:
        candidate.relative_to(PROJECT_ROOT.resolve())
    except ValueError:
        return None
    return candidate


execution_sidecar = read_json(EXECUTION_SIDECAR_PATH)
lineage_data = read_json(LINEAGE_PATH)
verified_execution_status = read_json(VERIFIED_EXECUTION_STATUS_PATH)
original_snapshot_info = lineage_data.get('original_snapshot', {})
EXECUTION_CANDIDATE_PATH = safe_project_path(original_snapshot_info.get('path'))
verified_path_text = verified_execution_status.get('verified_output_path') or lineage_data.get('verified_dataset', {}).get('path')
VERIFIED_EXECUTION_PATH = safe_project_path(verified_path_text)
verified_manifest_text = verified_execution_status.get('manifest_path') or lineage_data.get('verified_dataset', {}).get('manifest_path')
VERIFIED_EXECUTION_MANIFEST_PATH = safe_project_path(verified_manifest_text)
immutable_copy_candidates = list((BACKTEST_DIR / 'execution_data_v2').glob('*original*.parquet'))
IMMUTABLE_COPY_PATH = immutable_copy_candidates[0] if len(immutable_copy_candidates) == 1 else None

existing_gate_checklist = read_csv_if_present(GATE_CHECKLIST_PATH)
execution_price_audit = read_csv_if_present(EXECUTION_AUDIT_PATH)
corporate_action_ledger = read_csv_if_present(CORPORATE_ACTION_LEDGER_PATH)
corporate_action_vendor_events = read_csv_if_present(CORPORATE_ACTION_VENDOR_PATH)
banpu_audit = read_csv_if_present(BANPU_AUDIT_PATH)
global_tick_audit = read_csv_if_present(GLOBAL_TICK_AUDIT_PATH)
global_recent_crosscheck = read_csv_if_present(GLOBAL_RECENT_CROSSCHECK_PATH)
security_identity_mapping = read_csv_if_present(SECURITY_IDENTITY_PATH)
execution_tick_audit = read_csv_if_present(EXECUTION_TICK_AUDIT_PATH)
execution_tick_original_audit = read_csv_if_present(EXECUTION_TICK_ORIGINAL_PATH)
verified_execution_manifest = read_json(VERIFIED_EXECUTION_MANIFEST_PATH) if VERIFIED_EXECUTION_MANIFEST_PATH else {}

feature_path = FEATURE_MASTER_PATH if FEATURE_MASTER_PATH.is_file() else FEATURE_MASTER_FALLBACK_PATH
feature_df = pd.read_parquet(feature_path) if feature_path.is_file() else pd.DataFrame()
# Snapshot นี้เป็น vendor candidate เพื่อ audit เท่านั้น ไม่ได้อนุมัติเป็น Execution OHLCV
execution_candidate_df = pd.read_parquet(EXECUTION_CANDIDATE_PATH) if EXECUTION_CANDIDATE_PATH and EXECUTION_CANDIDATE_PATH.is_file() else pd.DataFrame()
if VERIFIED_EXECUTION_PATH and VERIFIED_EXECUTION_PATH.is_file():
    if VERIFIED_EXECUTION_PATH.suffix.lower() == '.parquet':
        verified_execution_df = pd.read_parquet(VERIFIED_EXECUTION_PATH)
    elif VERIFIED_EXECUTION_PATH.suffix.lower() == '.csv':
        verified_execution_df = pd.read_csv(VERIFIED_EXECUTION_PATH)
    else:
        raise ValueError(f'Unsupported verified execution format: {VERIFIED_EXECUTION_PATH.suffix}')
else:
    verified_execution_df = pd.DataFrame()

print('Original snapshot path from lineage:', EXECUTION_CANDIDATE_PATH)
print('Original snapshot SHA-256:', sha256_file(EXECUTION_CANDIDATE_PATH) if EXECUTION_CANDIDATE_PATH and EXECUTION_CANDIDATE_PATH.is_file() else 'MISSING')
print('Verified execution dataset exists:', bool(VERIFIED_EXECUTION_PATH and VERIFIED_EXECUTION_PATH.is_file()))
print('Feature shape:', feature_df.shape, '| Execution candidate shape:', execution_candidate_df.shape)
#

Original snapshot path from lineage: D:\Project_Github\set_fear_index\dataset\verified_build_20261113_152948\SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.parquet
Original snapshot SHA-256: 14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a
Verified execution dataset exists: False
Feature shape: (198532, 17) | Execution candidate shape: (17430, 10)


## 5. Feature / Execution integrity audits

ตรวจ schema, key ซ้ำ, missingness และ OHLC โดยแยก Execution candidate จาก verified execution data

In [17]:
# สรุปข้อเท็จจริงจากไฟล์ที่โหลด โดยไม่แก้ DataFrame ต้นฉบับ
def data_health(frame: pd.DataFrame, required: Sequence[str], key: Sequence[str]) -> dict:
    if frame.empty:
        return {'exists': False, 'rows': 0, 'missing_columns': list(required)}
    missing_columns = sorted(set(required) - set(frame.columns))
    return {
        'exists': True,
        'rows': int(len(frame)),
        'missing_columns': missing_columns,
        'duplicate_keys': int(frame.duplicated(list(key)).sum()) if not missing_columns else None,
    }


feature_required = ['Date', 'Symbol', *FEATURE_COLUMNS]
execution_required = ['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']
feature_health = data_health(feature_df, feature_required, ['Date', 'Symbol'])
execution_candidate_health = data_health(execution_candidate_df, execution_required, ['Date', 'Symbol'])
verified_execution_health = data_health(verified_execution_df, execution_required, ['Date', 'Symbol'])
feature_missing_counts = {column: int(feature_df[column].isna().sum()) for column in FEATURE_COLUMNS if column in feature_df}
feature_target_present = TARGET_COLUMN in feature_df.columns
feature_2026 = feature_df.loc[pd.to_datetime(feature_df['Date']).dt.year == 2026] if not feature_df.empty and 'Date' in feature_df else pd.DataFrame()

feature_impact_required = {'Date', 'Symbol', *FEATURE_COLUMNS}
missing_feature_impact_columns = sorted(feature_impact_required - set(feature_df.columns))
if missing_feature_impact_columns:
    raise ValueError(f'Cannot build feature impact audit; missing columns: {missing_feature_impact_columns}')

banpu_identity_start = None
if not security_identity_mapping.empty and {'Security_ID', 'Valid_From'}.issubset(security_identity_mapping.columns):
    banpu_new_identity = security_identity_mapping.loc[
        security_identity_mapping['Security_ID'].astype(str).str.upper().eq('BANPU_NEW'), 'Valid_From'
    ]
    if not banpu_new_identity.empty:
        banpu_identity_start = pd.to_datetime(banpu_new_identity.iloc[0], errors='raise').normalize()

feature_impact_rows = []
for impact_symbol in ['GLOBAL', 'BANPU']:
    symbol_features = feature_df.loc[feature_df['Symbol'].astype(str).str.upper().eq(impact_symbol)].copy()
    symbol_dates = pd.to_datetime(symbol_features['Date'])
    feature_impact_rows.append({
        'Symbol': impact_symbol,
        'Feature_Rows': int(len(symbol_features)),
        'Feature_Rows_2026': int(symbol_dates.dt.year.eq(2026).sum()),
        'Rows_With_All_Six_Features': int(symbol_features[FEATURE_COLUMNS].notna().all(axis=1).sum()),
        'Rows_With_Target_Label_Audit_Only': int(symbol_features[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in symbol_features.columns else None,
        'BANPU_Post_2026_08_04_Rows': int((symbol_dates.ge(banpu_identity_start) & symbol_dates.dt.year.eq(2026)).sum()) if impact_symbol == 'BANPU' and banpu_identity_start is not None else None,
        'Impact_Status': 'BLOCKED: adjusted-price/identity lineage and rebuilt-feature comparison unavailable',
    })
feature_impact_audit = pd.DataFrame(feature_impact_rows)

candidate_ohlc_missing = {}
candidate_zero_volume = None
candidate_invalid_ohlc = None
if not execution_candidate_df.empty and set(['Open', 'High', 'Low', 'Close', 'Volume']).issubset(execution_candidate_df.columns):
    candidate_ohlc_missing = {column: int(execution_candidate_df[column].isna().sum()) for column in ['Open', 'High', 'Low', 'Close']}
    candidate_zero_volume = int((execution_candidate_df['Volume'].fillna(0) <= 0).sum())
    valid_ohlc = execution_candidate_df[['Open', 'High', 'Low', 'Close']].notna().all(axis=1)
    candidate_invalid_ohlc = int((valid_ohlc & (
        (execution_candidate_df['Low'] > execution_candidate_df['High']) |
        (execution_candidate_df['Open'] < execution_candidate_df['Low']) |
        (execution_candidate_df['Open'] > execution_candidate_df['High']) |
        (execution_candidate_df['Close'] < execution_candidate_df['Low']) |
        (execution_candidate_df['Close'] > execution_candidate_df['High'])
    )).sum())


def invalid_ohlc_rows(frame: pd.DataFrame) -> int | None:
    required = {'Open', 'High', 'Low', 'Close', 'Volume'}
    if frame.empty or not required.issubset(frame.columns):
        return None
    prices = frame[['Open', 'High', 'Low', 'Close']]
    complete = prices.notna().all(axis=1) & frame['Volume'].notna()
    invalid_prices = (
        (prices <= 0).any(axis=1)
        | (frame['Low'] > frame['High'])
        | (frame['Open'] < frame['Low'])
        | (frame['Open'] > frame['High'])
        | (frame['Close'] < frame['Low'])
        | (frame['Close'] > frame['High'])
        | (frame['Volume'] < 0)
    )
    return int((complete & invalid_prices).sum())


verified_invalid_ohlc = invalid_ohlc_rows(verified_execution_df)

print('Feature health:', feature_health)
print('Feature missing counts:', feature_missing_counts)
print('Feature 2026 rows:', len(feature_2026), '| Actual target column present for audit only:', feature_target_present)
print('Execution candidate health:', execution_candidate_health)
print('Verified execution health:', verified_execution_health)
print('Candidate missing OHLC:', candidate_ohlc_missing, '| zero-volume rows:', candidate_zero_volume, '| OHLC range violations:', candidate_invalid_ohlc)
#

Feature health: {'exists': True, 'rows': 198532, 'missing_columns': [], 'duplicate_keys': 0}
Feature missing counts: {'daily_return': 18, 'momentum_20d': 364, 'vol_z': 1380, 'VolRank': 79368, 'CorrRank': 82017, 'LiquidityRank': 79365}
Feature 2026 rows: 17367 | Actual target column present for audit only: True
Execution candidate health: {'exists': True, 'rows': 17430, 'missing_columns': [], 'duplicate_keys': 0}
Verified execution health: {'exists': False, 'rows': 0, 'missing_columns': ['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']}
Candidate missing OHLC: {'Open': 63, 'High': 63, 'Low': 63, 'Close': 64} | zero-volume rows: 92 | OHLC range violations: 0


## 6. Corporate Action / GLOBAL / BANPU / Tick audits และ Feature impact

Vendor/corrected candidate เป็น diagnostic เท่านั้น; ไม่มีการปัดราคา/เติมราคา หรืออ้างว่าเป็นราคา SET ที่ยืนยันแล้ว

## 7. Frozen Model Artifact verification (no fit/refit)

In [18]:
# ค้นหา artifact ภายใน repository และจับคู่กับ metadata ที่ระบุ Original Softmax
model_notebook_path = PROJECT_ROOT / '04_model.ipynb'
model_notebook = json.loads(model_notebook_path.read_text(encoding='utf-8')) if model_notebook_path.is_file() else {}
model_notebook_source = '\n'.join(
    ''.join(cell.get('source', []))
    for cell in model_notebook.get('cells', [])
    if cell.get('cell_type') == 'code'
)
model_saved_from_selected_pipeline = bool(
    'final_model = grid_search.best_estimator_' in model_notebook_source
    and 'joblib.dump(final_model, model_path)' in model_notebook_source
    and 'final_softmax_model.joblib' in model_notebook_source
)
MODEL_SUFFIXES = {'.joblib', '.pkl', '.pickle', '.sav'}
model_artifact_candidates = []
models_root = PROJECT_ROOT / 'models'
if models_root.is_dir():
    for candidate in models_root.rglob('*'):
        if candidate.is_file() and candidate.suffix.lower() in MODEL_SUFFIXES:
            metadata_candidate = candidate.parent / 'final_softmax_metadata.json'
            metadata = read_json(metadata_candidate)
            if metadata.get('model_name') == 'Original Softmax' and metadata_candidate.is_file():
                model_artifact_candidates.append((candidate, metadata_candidate, metadata))

MODEL_ARTIFACT_PATH = None
MODEL_METADATA_PATH = None
model_metadata = {}
loaded_model = None
model_check_details = {}
if len(model_artifact_candidates) == 1:
    MODEL_ARTIFACT_PATH, MODEL_METADATA_PATH, model_metadata = model_artifact_candidates[0]
    loaded_model = joblib.load(MODEL_ARTIFACT_PATH)
    actual_steps = [name for name, _ in loaded_model.steps] if isinstance(loaded_model, Pipeline) else []
    logistic = loaded_model.named_steps.get('model') if isinstance(loaded_model, Pipeline) else None
    metadata_features = list(model_metadata.get('features', []))
    metadata_class_weight = model_metadata.get('best_parameters', {}).get('model__class_weight')
    if isinstance(metadata_class_weight, dict):
        metadata_class_weight = {int(key): value for key, value in metadata_class_weight.items()}
    model_features = list(getattr(loaded_model, 'feature_names_in_', []))
    classes = list(getattr(loaded_model, 'classes_', []))
    model_check_details = {
        'single_repository_artifact_with_metadata': True,
        'selected_pipeline_save_code': model_saved_from_selected_pipeline,
        'pipeline_type': isinstance(loaded_model, Pipeline),
        'pipeline_steps': actual_steps,
        'feature_order_matches': model_features == metadata_features == FEATURE_COLUMNS,
        'class_order_matches': classes == [0, 1, 2],
        'scaler_type': isinstance(loaded_model.named_steps.get('scaler'), StandardScaler) if isinstance(loaded_model, Pipeline) else False,
        'classifier_type': isinstance(logistic, LogisticRegression),
        'parameters_match_metadata': bool(
            isinstance(logistic, LogisticRegression)
            and logistic.C == model_metadata.get('best_parameters', {}).get('model__C')
            and logistic.class_weight == metadata_class_weight
            and logistic.l1_ratio == model_metadata.get('best_parameters', {}).get('model__l1_ratio')
        ),
        'training_period': model_metadata.get('training_period'),
        'validation_period': model_metadata.get('validation_period'),
        'test_period': model_metadata.get('test_period'),
        'training_samples': model_metadata.get('training_samples'),
        'validation_samples': model_metadata.get('validation_samples'),
        'test_samples': model_metadata.get('test_samples'),
    }
    MODEL_ARTIFACT_VERIFIED = all([
        model_check_details['selected_pipeline_save_code'], model_check_details['pipeline_type'],
        actual_steps == ['scaler', 'model'], model_check_details['feature_order_matches'],
        model_check_details['class_order_matches'], model_check_details['scaler_type'],
        model_check_details['classifier_type'], model_check_details['parameters_match_metadata'],
        model_metadata.get('prediction_rule') == 'Argmax',
    ])
else:
    MODEL_ARTIFACT_VERIFIED = False
    model_check_details = {'repository_artifact_matches': len(model_artifact_candidates)}

print('Model artifact:', MODEL_ARTIFACT_PATH)
print('Model SHA-256:', sha256_file(MODEL_ARTIFACT_PATH) if MODEL_ARTIFACT_PATH else None)
print('Model metadata:', MODEL_METADATA_PATH, '| test metrics:', model_metadata.get('test_metrics'))
print('Artifact checks:', model_check_details, '| verified:', MODEL_ARTIFACT_VERIFIED)
#

Model artifact: D:\Project_Github\set_fear_index\models\Original_Softmax_Final\final_softmax_model.joblib
Model SHA-256: 8ec9433bd5fa6419b7f6870f1058917b7fb3f6f0cb66cbc771a45f6956a048c7
Model metadata: D:\Project_Github\set_fear_index\models\Original_Softmax_Final\final_softmax_metadata.json | test metrics: {'accuracy': 0.8305, 'macro_f1': 0.3936, 'balanced_accuracy': 0.3932, 'log_loss': 0.7459}
Artifact checks: {'single_repository_artifact_with_metadata': True, 'selected_pipeline_save_code': True, 'pipeline_type': True, 'pipeline_steps': ['scaler', 'model'], 'feature_order_matches': True, 'class_order_matches': True, 'scaler_type': True, 'classifier_type': True, 'parameters_match_metadata': True, 'training_period': '2016-2024', 'validation_period': '2025', 'test_period': '2026', 'training_samples': 94205, 'validation_samples': 11824, 'test_samples': 10436} | verified: True


## 8. Chronological Prediction และ Trading Calendar (`t → t+1`)

Prediction รับเฉพาะ Feature ตามลำดับที่ฝึกไว้; ไม่ส่ง Target, label หรือข้อมูลหลัง decision date เข้าโมเดล

In [19]:
# สร้าง Prediction ทีละ decision date เพื่อบังคับให้ input เป็น Feature ของวันเดียว
def generate_predictions(feature_rows: pd.DataFrame, frozen_model) -> pd.DataFrame:
    required = {'Date', 'Symbol', *FEATURE_COLUMNS}
    missing = sorted(required - set(feature_rows.columns))
    if missing:
        raise ValueError(f'Feature rows ขาดคอลัมน์: {missing}')
    if 'apply_approved_exclusions' not in globals():
        raise RuntimeError('Approved exclusion policy must be loaded before feature inference')
    feature_rows = apply_approved_exclusions(feature_rows, 'Feature inputs')
    if feature_rows.empty:
        raise ValueError('Feature inputs contain no non-excluded symbols')
    dates = pd.to_datetime(feature_rows['Date']).dt.normalize().unique()
    if len(dates) != 1:
        raise ValueError('ต้องส่ง Feature ของ decision date เดียวต่อการสร้าง Prediction หนึ่งครั้ง')
    features_only = feature_rows.loc[:, FEATURE_COLUMNS].copy()
    if features_only.isna().any(axis=None):
        raise ValueError('Feature มี missing value; ห้ามเติม/แทนค่าด้วยข้อมูลอื่นโดยไม่มี policy')
    predicted = np.asarray(frozen_model.predict(features_only))
    probabilities = np.asarray(frozen_model.predict_proba(features_only))
    if list(getattr(frozen_model, 'classes_', [])) != [0, 1, 2] or probabilities.shape != (len(feature_rows), 3):
        raise ValueError('Model classes/probability shape ไม่ตรงกับ frozen class mapping')
    result = feature_rows.loc[:, ['Date', 'Symbol']].copy()
    result['Predicted_Class'] = predicted.astype(int)
    result['Prediction_Label'] = result['Predicted_Class'].map(TARGET_LABELS)
    for index, label in enumerate(['Panic', 'FOMO', 'Normal']):
        result[f'P_{label}'] = probabilities[:, index]
    return result


def build_next_session_map(validated_sessions: Sequence) -> dict[pd.Timestamp, pd.Timestamp]:
    dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(list(validated_sessions))).dropna().unique()).normalize().sort_values()
    if dates.has_duplicates:
        raise ValueError('ปฏิทินมีวันที่ซ้ำ')
    return {dates[index]: dates[index + 1] for index in range(len(dates) - 1)}


def align_decision_to_execution(decision_date, next_session_map: Mapping) -> pd.Timestamp:
    decision = pd.Timestamp(decision_date).normalize()
    if decision not in next_session_map:
        raise KeyError(f'ไม่มี validated next session สำหรับ {decision.date()}')
    execution = pd.Timestamp(next_session_map[decision]).normalize()
    if execution <= decision:
        raise ValueError('execution session ต้องอยู่หลัง decision date')
    return execution


def build_membership_events(constituents_path: Path, declarations_path: Path) -> pd.DataFrame:
    if not constituents_path.is_file() or not declarations_path.is_file():
        raise FileNotFoundError('ต้องมี constituent และ effective-date files')
    constituents = pd.read_csv(constituents_path, dtype=str, keep_default_na=False)
    declarations = pd.read_csv(declarations_path, dtype={'year': str, 'review_period': str})
    required_constituents = {'period'}
    required_declarations = {'year', 'review_period', 'effective_date'}
    if not required_constituents.issubset(constituents.columns) or not required_declarations.issubset(declarations.columns):
        raise ValueError('Membership schema ไม่ครบ')
    member_columns = [column for column in constituents if column.startswith('symbol_')]
    if not member_columns:
        raise ValueError('ไม่พบ constituent symbol columns')
    constituents['period'] = constituents['period'].astype(str).str.strip()
    declarations['period'] = (
        declarations['year'].astype(str).str.strip() + '_'
        + declarations['review_period'].astype(str).str.strip().str.upper()
    )
    declarations['effective_date'] = pd.to_datetime(declarations['effective_date'], errors='raise').dt.normalize()
    periods = constituents.merge(
        declarations[['period', 'effective_date', *(['source_url'] if 'source_url' in declarations else [])]],
        on='period', how='left', validate='one_to_one',
    ).sort_values(['effective_date', 'period'])
    if periods['effective_date'].isna().any() or periods['effective_date'].duplicated().any():
        raise ValueError('Membership effective dates are missing or duplicated')
    events = []
    previous = set()
    for row in periods.to_dict('records'):
        values = [str(row[column]).strip().upper() for column in member_columns if str(row[column]).strip()]
        current = set(values)
        if len(current) != len(values):
            raise ValueError(f'Duplicate member symbol in period {row["period"]}')
        effective = pd.Timestamp(row['effective_date'])
        events.extend({'Symbol': symbol, 'effective_date': effective, 'action': 'ADD'} for symbol in sorted(current - previous))
        events.extend({'Symbol': symbol, 'effective_date': effective, 'action': 'REMOVE'} for symbol in sorted(previous - current))
        previous = current
    return pd.DataFrame(events, columns=['Symbol', 'effective_date', 'action']).sort_values(
        ['effective_date', 'Symbol']
    ).reset_index(drop=True)


def build_membership_by_date(events: pd.DataFrame, dates: Sequence) -> dict[pd.Timestamp, set[str]]:
    ordered_events = events.sort_values(['effective_date', 'Symbol']).to_dict('records')
    session_dates = sorted(pd.to_datetime(pd.Series(list(dates))).dt.normalize().unique())
    state, result, position = {}, {}, 0
    for current_date in session_dates:
        current_date = pd.Timestamp(current_date)
        while position < len(ordered_events) and pd.Timestamp(ordered_events[position]['effective_date']) <= current_date:
            event = ordered_events[position]
            state[event['Symbol']] = event['action'] == 'ADD'
            position += 1
        result[current_date] = {symbol for symbol, active in state.items() if active}
    return result


def generate_signals(predictions: pd.DataFrame, strategy_name: str, strategy_map: Mapping) -> pd.DataFrame:
    forbidden = {TARGET_COLUMN, 'Actual_Target', 'target_aligned', 'y_test'}
    if forbidden.intersection(predictions.columns):
        raise ValueError('ห้ามส่ง Actual Target เข้า Trading Signal path')
    if strategy_name not in strategy_map:
        raise KeyError(f'ไม่รู้จัก strategy: {strategy_name}')
    result = predictions.loc[:, ['Date', 'Symbol', 'Predicted_Class', 'Prediction_Label']].copy()
    result['Signal'] = result['Prediction_Label'].map(strategy_map[strategy_name])
    if result['Signal'].isna().any():
        raise ValueError('มี Prediction label ที่ไม่อยู่ใน frozen strategy map')
    return result


# Feature dates เป็น observed candidate calendar เท่านั้น ไม่ใช่ official SET calendar
observed_feature_dates = pd.DatetimeIndex(pd.to_datetime(feature_df['Date']).dropna().unique()).sort_values() if not feature_df.empty and 'Date' in feature_df else pd.DatetimeIndex([])
membership_events = build_membership_events(MEMBERSHIP_PATH, DECLARATION_PATH)
observed_membership_by_date = build_membership_by_date(membership_events, observed_feature_dates)
membership_declarations = pd.read_csv(DECLARATION_PATH) if DECLARATION_PATH.is_file() else pd.DataFrame()
membership_source_urls_present = bool(
    membership_declarations['source_url'].astype('string').fillna('').str.match(r'https?://').all()
) if not membership_declarations.empty and 'source_url' in membership_declarations else False
membership_source_urls_official = bool(
    membership_declarations['source_url'].astype('string').fillna('').str.startswith('https://www.set.or.th/').all()
) if membership_source_urls_present else False
validated_session_calendar = None
print('Observed feature dates:', len(observed_feature_dates), '| membership events:', len(membership_events))
print('Membership evidence URLs present:', membership_source_urls_present, '| all primary SET URLs:', membership_source_urls_official)
print('Observed membership map is candidate only; official validated calendar available:', validated_session_calendar is not None)
#

Observed feature dates: 2638 | membership events: 214
Membership evidence URLs present: True | all primary SET URLs: False
Observed membership map is candidate only; official validated calendar available: False


## 9. Strategy / Allocation configuration

แสดง 9 × 7 ขอบเขตการทดลองเท่านั้น; ไม่ได้หมายความว่า Allocation ทุกวิธีพร้อมใช้หรือ Gate ผ่าน

In [20]:
# Frozen Strategy definitions และ Cartesian product สำหรับตรวจจำนวน combination
STRATEGY_MAP = {
    'S1': {'Panic': 'Buy', 'FOMO': 'Buy', 'Normal': 'Hold'},
    'S2': {'Panic': 'Buy', 'FOMO': 'Sell', 'Normal': 'Hold'},
    'S3': {'Panic': 'Buy', 'FOMO': 'Hold', 'Normal': 'Hold'},
    'S4': {'Panic': 'Sell', 'FOMO': 'Buy', 'Normal': 'Hold'},
    'S5': {'Panic': 'Sell', 'FOMO': 'Sell', 'Normal': 'Hold'},
    'S6': {'Panic': 'Sell', 'FOMO': 'Hold', 'Normal': 'Hold'},
    'S7': {'Panic': 'Hold', 'FOMO': 'Buy', 'Normal': 'Hold'},
    'S8': {'Panic': 'Hold', 'FOMO': 'Sell', 'Normal': 'Hold'},
    'S9': {'Panic': 'Hold', 'FOMO': 'Hold', 'Normal': 'Hold'},
}
ALLOCATION_METHODS = ['EW', 'IV', 'ERC', 'HRP', 'MinVar', 'CVaR', 'MPT']
EXPERIMENT_COMBINATIONS = list(product(STRATEGY_MAP, ALLOCATION_METHODS))
assert len(STRATEGY_MAP) == 9 and len(ALLOCATION_METHODS) == 7 and len(EXPERIMENT_COMBINATIONS) == 63


EXCLUSION_POLICY_PATH = BACKTEST_DIR / 'research_controls' / 'execution_excluded_symbols.csv'


def load_approved_exclusions(path: Path) -> frozenset[str]:
    required = {
        'Symbol', 'Exclusion_Start_Date', 'Exclusion_End_Date',
        'Exclusion_Reason', 'Decision_Status',
    }
    if not path.is_file():
        raise FileNotFoundError(f'ไม่พบ exclusion policy ที่ตรึงไว้: {path}')
    policy = pd.read_csv(path)
    if not required.issubset(policy.columns):
        raise ValueError(f'Exclusion policy ขาดคอลัมน์: {sorted(required - set(policy.columns))}')
    policy['Symbol'] = policy['Symbol'].astype('string').str.strip().str.upper()
    policy['Exclusion_Start_Date'] = pd.to_datetime(policy['Exclusion_Start_Date'], errors='raise')
    policy['Exclusion_End_Date'] = pd.to_datetime(policy['Exclusion_End_Date'], errors='raise')
    approved = policy.loc[policy['Decision_Status'].eq('APPROVED_RESEARCH_EXCLUSION')].copy()
    if approved.empty or approved['Symbol'].isna().any() or approved['Symbol'].duplicated().any():
        raise ValueError('Exclusion policy ต้องมี Symbol ที่อนุมัติ ไม่ซ้ำ และระบุครบถ้วน')
    if 'BANPU' not in set(approved['Symbol'].astype(str)):
        raise ValueError('BANPU must remain approved excluded for the full 2026 backtest')
    backtest_start = pd.Timestamp('2026-01-01')
    backtest_end = pd.Timestamp('2026-12-31')
    covers_year = (
        approved['Exclusion_Start_Date'].le(backtest_start)
        & approved['Exclusion_End_Date'].ge(backtest_end)
    )
    if not covers_year.all():
        raise ValueError('Approved exclusions ต้องครอบคลุมปี Backtest 2026 ทั้งปี')
    return frozenset(approved['Symbol'].astype(str))


EXCLUDED_SYMBOLS = load_approved_exclusions(EXCLUSION_POLICY_PATH)


def apply_approved_exclusions(frame: pd.DataFrame, context: str) -> pd.DataFrame:
    if 'Symbol' not in frame.columns:
        raise ValueError(f'{context} ไม่มีคอลัมน์ Symbol')
    symbols = frame['Symbol'].astype('string').str.strip().str.upper()
    if symbols.isna().any() or symbols.eq('').any():
        raise ValueError(f'{context} มี Symbol ว่าง')
    return frame.loc[~symbols.isin(EXCLUDED_SYMBOLS)].copy()


class PolicyUnresolvedError(RuntimeError):
    pass


def apply_weight_cap(raw_weights: pd.Series, cap: float = MAX_BUY_WEIGHT_PER_STOCK) -> pd.Series:
    if not 0 < cap <= 1:
        raise ValueError('cap ต้องอยู่ในช่วง (0, 1]')
    weights = pd.to_numeric(raw_weights, errors='raise').astype(float)
    if weights.index.has_duplicates or (weights < 0).any() or not np.isfinite(weights.to_numpy()).all():
        raise ValueError('น้ำหนักต้องมี index ไม่ซ้ำ เป็น finite และไม่ติดลบ')
    capped = pd.Series(0.0, index=weights.index, dtype=float)
    total_weight = float(weights.sum())
    if total_weight > 1.0 + 1e-12:
        raise ValueError('น้ำหนักรวมเกิน 1; อาจทำให้เกิด leverage')
    if total_weight <= 0:
        return capped
    active = list(weights.index[weights > 0])
    remaining = total_weight
    while active and remaining > 1e-12:
        active_total = float(weights.loc[active].sum())
        if active_total <= 0:
            break
        proposed = weights.loc[active] / active_total * remaining
        over_cap = list(proposed.index[proposed > cap + 1e-12])
        if not over_cap:
            capped.loc[proposed.index] = proposed
            break
        for symbol in over_cap:
            capped.loc[symbol] = cap
            remaining -= cap
            active.remove(symbol)
    return capped


def calculate_allocation_weights(method: str, symbols: Sequence[str], historical_returns: pd.DataFrame | None = None) -> pd.Series:
    names = list(dict.fromkeys(str(symbol) for symbol in symbols))
    if not names:
        return pd.Series(dtype=float)
    if method == 'EW':
        raw = pd.Series(1.0, index=names)
    elif method == 'IV':
        if historical_returns is None or len(historical_returns) < ALLOCATION_LOOKBACK['IV']:
            raise PolicyUnresolvedError('IV ต้องมี 60 validated sessions; minimum-history fallback ยังไม่อนุมัติ')
        volatility = historical_returns.loc[:, names].tail(ALLOCATION_LOOKBACK['IV']).std(ddof=1)
        if volatility.isna().any() or (volatility <= 0).any():
            raise PolicyUnresolvedError('IV มี volatility ไม่พร้อม/เป็นศูนย์; fallback policy ยังไม่อนุมัติ')
        raw = 1.0 / volatility
    else:
        raise PolicyUnresolvedError(f'{method} ยังไม่มี allocator ที่อนุมัติ/ยืนยันครบ; ไม่สร้างน้ำหนักแทน')
    return apply_weight_cap(raw)


print('Strategies:', len(STRATEGY_MAP), '| allocations:', len(ALLOCATION_METHODS), '| combinations:', len(EXPERIMENT_COMBINATIONS))
#

Strategies: 9 | allocations: 7 | combinations: 63


## 10. Execution Engine mechanics (Raw Open only)

Pure functions สำหรับ synthetic tests; Research Engine จะเรียกได้ก็ต่อเมื่อ Data Integrity Gate และ policies ผ่าน

In [21]:
# Fill quote: slippage อยู่ใน Fill Price; fee คิดจาก Fill Notional เพียงหนึ่งครั้ง
@dataclass(frozen=True)
class FillQuote:
    side: str
    raw_open: float
    effective_price: float
    shares: int
    raw_notional: float
    gross_notional: float
    transaction_cost: float
    slippage_cost: float


def make_fill_quote(side: str, raw_open: float, shares: int) -> FillQuote:
    side = str(side).lower()
    if side not in {'buy', 'sell'}:
        raise ValueError('side ต้องเป็น buy หรือ sell')
    if not np.isfinite(raw_open) or raw_open <= 0:
        raise ValueError('Raw Open ต้องเป็นค่าบวกและ finite')
    if isinstance(shares, bool) or not isinstance(shares, (int, np.integer)) or shares <= 0:
        raise ValueError('shares ต้องเป็นจำนวนเต็มบวก')
    effective = float(raw_open) * (1.0 + SLIPPAGE_RATE if side == 'buy' else 1.0 - SLIPPAGE_RATE)
    fee_rate = BUY_COST_RATE if side == 'buy' else SELL_COST_RATE
    raw_notional = float(raw_open) * int(shares)
    gross_notional = effective * int(shares)
    return FillQuote(side, float(raw_open), effective, int(shares), raw_notional, gross_notional,
                     gross_notional * fee_rate, abs(effective - float(raw_open)) * int(shares))


def _order_record(decision_date, execution_date, symbol, action, reason, prediction_class=None,
                  raw_open=np.nan, effective_price=np.nan, shares=0, raw_notional=0.0,
                  gross_notional=0.0, fee=0.0, slippage=0.0, status='NO_FILL',
                  no_fill_reason=None, pre_cost_shares=None):
    pre_cost_shares = int(shares if pre_cost_shares is None else pre_cost_shares)
    return {
        'Decision_Date': pd.Timestamp(decision_date), 'Execution_Date': pd.Timestamp(execution_date),
        'Symbol': str(symbol), 'Action': str(action), 'Reason': str(reason), 'Prediction_Class': prediction_class,
        'Raw_Open': raw_open, 'Effective_Fill_Price': effective_price,
        'Pre_Cost_Shares': pre_cost_shares, 'Shares': int(shares),
        'Pre_Cost_Notional': float(raw_open) * pre_cost_shares if np.isfinite(raw_open) else 0.0,
        'Raw_Notional': float(raw_notional),
        'Gross_Notional': float(gross_notional), 'Fee': float(fee), 'Slippage_Cost': float(slippage),
        'Status': status, 'No_Fill_Reason': no_fill_reason,
    }


def execute_session(decision_date, execution_date, signals: Mapping[str, str], raw_open: Mapping[str, float],
                    execution_eligible: Mapping[str, bool], holdings: Mapping[str, int], cash: float,
                    target_weights: pd.Series, board_lot_size: int, set50_members: set[str] | None = None,
                    prediction_classes: Mapping[str, str] | None = None) -> tuple[float, dict[str, int], list[dict]]:
    if isinstance(board_lot_size, bool) or not isinstance(board_lot_size, (int, np.integer)) or board_lot_size <= 0:
        raise PolicyUnresolvedError('Board lot ต้องเป็นค่าที่ผ่านการยืนยัน; ห้ามเดา')
    cash = float(cash)
    if not np.isfinite(cash) or cash < -1e-8:
        raise ValueError('cash ต้อง finite และไม่ติดลบ')
    positions = {str(symbol): int(shares) for symbol, shares in holdings.items() if int(shares) > 0}
    if any(shares < 0 for shares in positions.values()):
        raise ValueError('จำนวนหุ้นต้องไม่ติดลบ')
    if any(action not in {'Buy', 'Sell', 'Hold'} for action in signals.values()):
        raise ValueError('Signal ต้องเป็น Buy/Sell/Hold')
    orders = []
    classes = prediction_classes or {}
    forced = {symbol for symbol in positions if set50_members is not None and symbol not in set50_members}
    sells = {symbol for symbol, action in signals.items() if action == 'Sell' and symbol in positions} | forced

    # ขายสัญญาณและ forced exit ก่อน; เงินจาก Sell ที่ไม่ Fill ใช้ต่อไม่ได้
    for symbol in sorted(sells):
        shares = positions.get(symbol, 0)
        reason = 'forced_set50_exit' if symbol in forced else 'signal_sell'
        raw = raw_open.get(symbol, np.nan)
        eligible = execution_eligible.get(symbol, False)
        if not eligible or not np.isfinite(raw) or raw <= 0:
            why = 'not_tradable_or_missing_open' if not eligible else 'missing_or_invalid_open'
            orders.append(_order_record(decision_date, execution_date, symbol, 'Sell', reason, classes.get(symbol), no_fill_reason=why))
            continue
        quote = make_fill_quote('sell', float(raw), shares)
        cash += quote.gross_notional - quote.transaction_cost
        positions.pop(symbol, None)
        orders.append(_order_record(decision_date, execution_date, symbol, 'Sell', reason, classes.get(symbol),
                                    quote.raw_open, quote.effective_price, shares, quote.raw_notional,
                                    quote.gross_notional, quote.transaction_cost, quote.slippage_cost, 'FILLED'))

    # Buy ใช้เฉพาะ cash หลัง Sell ที่ Fill จริง; Hold ไม่ถูกขายเพื่อหาเงิน
    buy_names = sorted(symbol for symbol, action in signals.items() if action == 'Buy')
    deployable_cash = cash
    unexpected_weights = set(target_weights.index) - set(buy_names)
    if unexpected_weights:
        raise ValueError(f'Allocator returned weights for non-Buy symbols: {sorted(unexpected_weights)}')
    weights = apply_weight_cap(target_weights) if len(target_weights) else pd.Series(dtype=float)
    for symbol in buy_names:
        raw = raw_open.get(symbol, np.nan)
        eligible = execution_eligible.get(symbol, False)
        weight = float(weights.get(symbol, 0.0))
        if not eligible:
            orders.append(_order_record(decision_date, execution_date, symbol, 'Buy', 'signal_buy', classes.get(symbol), no_fill_reason='not_tradable_or_missing_open'))
            continue
        if not np.isfinite(raw) or raw <= 0:
            orders.append(_order_record(decision_date, execution_date, symbol, 'Buy', 'signal_buy', classes.get(symbol), no_fill_reason='missing_or_invalid_open'))
            continue
        if weight <= 0:
            orders.append(_order_record(decision_date, execution_date, symbol, 'Buy', 'signal_buy', classes.get(symbol), no_fill_reason='no_allocator_weight'))
            continue
        budget = min(weight * deployable_cash, MAX_BUY_WEIGHT_PER_STOCK * deployable_cash, cash)
        unit_cost = float(raw) * (1.0 + SLIPPAGE_RATE) * (1.0 + BUY_COST_RATE)
        pre_cost_shares = int(math.floor((budget / float(raw)) / int(board_lot_size)) * int(board_lot_size))
        shares = int(math.floor((budget / unit_cost) / int(board_lot_size)) * int(board_lot_size))
        while shares > 0:
            quote = make_fill_quote('buy', float(raw), shares)
            if quote.gross_notional + quote.transaction_cost <= cash + 1e-8:
                break
            shares -= int(board_lot_size)
        if shares <= 0:
            orders.append(_order_record(decision_date, execution_date, symbol, 'Buy', 'signal_buy', classes.get(symbol), no_fill_reason='cash_or_board_lot_insufficient'))
            continue
        quote = make_fill_quote('buy', float(raw), shares)
        cash -= quote.gross_notional + quote.transaction_cost
        positions[symbol] = positions.get(symbol, 0) + shares
        orders.append(_order_record(decision_date, execution_date, symbol, 'Buy', 'signal_buy', classes.get(symbol),
                                    quote.raw_open, quote.effective_price, shares, quote.raw_notional,
                                    quote.gross_notional, quote.transaction_cost, quote.slippage_cost, 'FILLED',
                                    pre_cost_shares=pre_cost_shares))

    for symbol, action in sorted(signals.items()):
        if action == 'Hold' and symbol not in forced:
            orders.append(_order_record(decision_date, execution_date, symbol, 'Hold', 'signal_hold',
                                        classes.get(symbol), shares=positions.get(symbol, 0), status='NO_ACTION'))
    if cash < -1e-7 or any(shares < 0 for shares in positions.values()):
        raise AssertionError('cash/share invariant violated')
    return max(0.0, float(cash)), positions, orders
#

## 11. Portfolio Accounting, Corporate Actions, Valuation และ Performance Metrics

In [22]:
# Corporate Action ที่ไม่มี row-level official reconciliation จะไม่ปรับ Cash/Shares
class UnverifiedCorporateActionError(RuntimeError):
    pass


def apply_verified_corporate_actions(action_date, cash: float, holdings: Mapping[str, int], ledger: pd.DataFrame):
    required = {'Effective_Date', 'Reconciliation_Status', 'Official_Source'}
    if ledger.empty or not required.issubset(ledger.columns):
        raise UnverifiedCorporateActionError('ไม่มี Official Corporate Action ledger ที่ตรวจสอบได้')
    fully_reconciled = ledger['Reconciliation_Status'].astype(str).str.upper().eq('RECONCILED').all()
    all_sources_official = ledger['Official_Source'].astype('string').fillna('').str.match(r'https?://').all()
    if not fully_reconciled or not all_sources_official:
        raise UnverifiedCorporateActionError('Ledger มี Corporate Action ที่ยังไม่ reconcile; ห้ามอนุมานว่าไม่มี action ในวันนี้')
    dates = pd.to_datetime(ledger['Effective_Date'], errors='coerce').dt.normalize()
    if dates.isna().any():
        raise UnverifiedCorporateActionError('Ledger มี Effective_Date ที่ขาดหรือ parse ไม่ได้')
    rows = ledger.loc[dates == pd.Timestamp(action_date).normalize()]
    if rows.empty:
        return float(cash), dict(holdings)
    raise PolicyUnresolvedError('Split/dividend/merger/fractional-share accounting terms ยังไม่ได้อนุมัติและทดสอบ')


def mark_to_market(valuation_date, cash: float, holdings: Mapping[str, int],
                   raw_close_quotes: Mapping[str, tuple[float, object]],
                   max_stale_calendar_days: int | None = None) -> tuple[float, float, pd.DataFrame]:
    date = pd.Timestamp(valuation_date).normalize()
    cash = float(cash)
    if not np.isfinite(cash) or cash < -1e-8:
        raise ValueError('cash ต้อง finite และไม่ติดลบ')
    market_value = 0.0
    records = []
    for symbol, shares in sorted(holdings.items()):
        if shares <= 0:
            continue
        quote = raw_close_quotes.get(symbol)
        if quote is None:
            raise ValueError(f'Missing Raw Close valuation quote: {symbol} ณ {date.date()}')
        close_price, price_date = quote
        price_date = pd.Timestamp(price_date).normalize()
        if not np.isfinite(close_price) or close_price <= 0 or price_date > date:
            raise ValueError(f'Invalid Raw Close valuation quote: {symbol}')
        stale_days = int((date - price_date).days)
        if stale_days > 0 and (max_stale_calendar_days is None or stale_days > max_stale_calendar_days):
            raise ValueError(f'Stale valuation exceeds approved policy: {symbol}, {stale_days} days')
        value = int(shares) * float(close_price)
        market_value += value
        records.append({
            'Date': date, 'Symbol': symbol, 'Shares': int(shares), 'Valuation_Price': float(close_price),
            'Price_Date': price_date, 'Stale_Calendar_Days': stale_days, 'Is_Stale': stale_days > 0,
            'Market_Value': value,
        })
    return cash + market_value, market_value, pd.DataFrame(records)


def calculate_performance_metrics(daily_curve: pd.DataFrame, initial_equity: float, initial_date,
                                  transaction_log: pd.DataFrame | None = None,
                                  valuation_log: pd.DataFrame | None = None) -> dict:
    if initial_equity <= 0 or not np.isfinite(initial_equity):
        raise ValueError('initial_equity ต้องเป็นค่าบวก')
    if not {'Date', 'Portfolio_Value'}.issubset(daily_curve.columns) or daily_curve.empty:
        raise ValueError('daily_curve ต้องมี Date/Portfolio_Value และไม่ว่าง')
    ordered = daily_curve.loc[:, ['Date', 'Portfolio_Value']].copy()
    ordered['Date'] = pd.to_datetime(ordered['Date']).dt.normalize()
    ordered = ordered.sort_values('Date').drop_duplicates('Date', keep='last')
    baseline_date = pd.Timestamp(initial_date).normalize()
    if ordered['Date'].min() <= baseline_date:
        raise ValueError('initial_date ต้องมาก่อนวัน valuation แรก')
    values = pd.Series([float(initial_equity), *ordered['Portfolio_Value'].astype(float).tolist()],
                       index=pd.DatetimeIndex([baseline_date, *ordered['Date'].tolist()]))
    if not np.isfinite(values.to_numpy()).all() or (values <= 0).any():
        raise ValueError('Equity curve มีค่าไม่ถูกต้อง')
    returns = values.pct_change().dropna()
    elapsed_days = int((values.index[-1] - values.index[0]).days)
    years = elapsed_days / 365.25 if elapsed_days > 0 else np.nan
    total_return = float(values.iloc[-1] / initial_equity - 1.0)
    cagr = float((values.iloc[-1] / initial_equity) ** (1.0 / years) - 1.0) if years and years > 0 else np.nan
    daily_std = float(returns.std(ddof=1)) if len(returns) > 1 else np.nan
    volatility = daily_std * np.sqrt(252) if np.isfinite(daily_std) else np.nan
    excess_returns = returns - SHARPE_RISK_FREE_RATE_DAILY
    sharpe = float(excess_returns.mean() / daily_std * np.sqrt(252)) if np.isfinite(daily_std) and daily_std > 0 else np.nan
    downside_deviation = float(np.sqrt(np.square(np.minimum(returns.to_numpy(), 0.0)).mean())) if len(returns) else np.nan
    sortino = float(returns.mean() / downside_deviation * np.sqrt(252)) if np.isfinite(downside_deviation) and downside_deviation > 0 else np.nan
    max_drawdown = float((values / values.cummax() - 1.0).min())
    orders = transaction_log if transaction_log is not None else pd.DataFrame()
    filled = orders.loc[orders['Status'].eq('FILLED')] if not orders.empty and 'Status' in orders else pd.DataFrame()
    valuations = valuation_log if valuation_log is not None else pd.DataFrame()
    stale = valuations.loc[valuations['Is_Stale'].astype(bool)] if not valuations.empty and 'Is_Stale' in valuations else pd.DataFrame()
    return {
        'Total_Return': total_return, 'CAGR': cagr, 'Annualized_Volatility': volatility,
        'Sharpe': sharpe, 'Sortino': sortino, 'Max_Drawdown': max_drawdown,
        'Calmar': cagr / abs(max_drawdown) if max_drawdown < 0 and np.isfinite(cagr) else np.nan,
        'Average_Cash_Weight': float(daily_curve['Cash_Weight'].mean()) if 'Cash_Weight' in daily_curve else np.nan,
        'Transaction_Count': int(len(filled)),
        'Total_Transaction_Cost': float(filled['Fee'].sum() + filled['Slippage_Cost'].sum()) if not filled.empty else 0.0,
        'Total_Turnover': float(filled['Gross_Notional'].sum()) if not filled.empty else 0.0,
        'Stale_Valuation_Count': int(len(stale)),
        'Max_Stale_Calendar_Days': int(valuations['Stale_Calendar_Days'].max()) if not valuations.empty and 'Stale_Calendar_Days' in valuations else 0,
        'Filled_Order_Count': int(len(filled)),
        'Unfilled_Order_Count': int((orders['Status'] == 'NO_FILL').sum()) if not orders.empty and 'Status' in orders else 0,
        'Forced_Exit_Count': int((filled['Reason'] == 'forced_set50_exit').sum()) if not filled.empty and 'Reason' in filled else 0,
        'Stock_Days_Held': int(daily_curve['Holding_Count'].sum()) if 'Holding_Count' in daily_curve else 0,
    }
#

## 12. Gate-guarded Research Backtest runner

Runner ต้องใช้ verified raw data, validated calendar, membership map และ reviewed accounting/allocator callbacks; ไม่มี fallback ไป candidate data

In [23]:
IMPLEMENTED_ALLOCATORS = {'EW', 'IV'}


def run_single_backtest(strategy_name: str, allocation_name: str, predictions: pd.DataFrame,
                         verified_execution: pd.DataFrame, validated_sessions: Sequence,
                         membership_by_date: Mapping, initial_equity: float, initial_date,
                         allocation_function: Callable, corporate_action_handler: Callable,
                         board_lot_size: int | None, max_stale_calendar_days: int | None):
    if not BACKTEST_GATE_OPEN:
        raise RuntimeError('BACKTEST BLOCKED: Data Integrity Gate ยังปิด')
    if allocation_name not in IMPLEMENTED_ALLOCATORS:
        raise PolicyUnresolvedError(f'{allocation_name} allocator ยังไม่ผ่าน methodology review')
    if not callable(corporate_action_handler) or not callable(allocation_function):
        raise PolicyUnresolvedError('ต้องระบุ Corporate Action accounting และ allocator ที่ผ่าน review')
    if board_lot_size is None or max_stale_calendar_days is None:
        raise PolicyUnresolvedError('Board lot และ stale valuation policy ยังไม่ยืนยัน')
    required_execution = {'Date', 'Symbol', 'Open', 'Close', 'Volume'}
    required_predictions = {'Date', 'Symbol', 'Predicted_Class', 'Prediction_Label'}
    if not required_execution.issubset(verified_execution.columns):
        raise ValueError(f'Execution data ขาดคอลัมน์: {sorted(required_execution - set(verified_execution.columns))}')
    if not required_predictions.issubset(predictions.columns):
        raise ValueError('Prediction schema ไม่ครบ')
    if {TARGET_COLUMN, 'Actual_Target', 'target_aligned', 'y_test'}.intersection(predictions.columns):
        raise ValueError('ห้าม Actual Target เข้าสู่ Backtest Runner')
    predictions = apply_approved_exclusions(predictions, 'Predictions')
    verified_execution = apply_approved_exclusions(verified_execution, 'Execution data')
    next_sessions = build_next_session_map(validated_sessions)
    execution = verified_execution.copy()
    execution['Date'] = pd.to_datetime(execution['Date']).dt.normalize()
    execution['Symbol'] = execution['Symbol'].astype(str)
    lookup = {(row.Date, row.Symbol): row for row in execution.itertuples(index=False)}
    cash, holdings = float(initial_equity), {}
    daily_rows, order_rows, valuation_rows, prediction_rows = [], [], [], []
    for decision_date in sorted(pd.to_datetime(predictions['Date']).dt.normalize().unique()):
        execution_date = align_decision_to_execution(decision_date, next_sessions)
        day = predictions.loc[pd.to_datetime(predictions['Date']).dt.normalize() == decision_date].copy()
        day = apply_approved_exclusions(day, 'Decision-date predictions')
        if day['Symbol'].duplicated().any():
            raise ValueError(f'Duplicate predictions ณ {decision_date.date()}')
        signal_rows = generate_signals(day, strategy_name, STRATEGY_MAP)
        signals = signal_rows.set_index('Symbol')['Signal'].to_dict()
        classes = signal_rows.set_index('Symbol')['Prediction_Label'].to_dict()
        buys = [symbol for symbol, action in signals.items() if action == 'Buy']
        if EXCLUDED_SYMBOLS.intersection(buys):
            raise AssertionError('Excluded security reached allocation inputs')
        weights = allocation_function(allocation_name, buys, decision_date) if buys else pd.Series(dtype=float)
        if EXCLUDED_SYMBOLS.intersection(set(weights.index.astype(str))):
            raise AssertionError('Excluded security received allocation weight')
        if EXCLUDED_SYMBOLS.intersection(set(holdings)):
            raise AssertionError('Excluded security appeared in portfolio holdings')
        names = (set(signals) | set(holdings)) - EXCLUDED_SYMBOLS
        day_rows = {symbol: lookup.get((execution_date, symbol)) for symbol in names}
        opens = {symbol: getattr(row, 'Open', np.nan) if row is not None else np.nan for symbol, row in day_rows.items()}
        eligible = {
            symbol: bool(row is not None and pd.notna(getattr(row, 'Open', np.nan)) and row.Open > 0
                         and pd.notna(getattr(row, 'Volume', np.nan)) and row.Volume > 0)
            for symbol, row in day_rows.items()
        }
        cash, holdings = corporate_action_handler(execution_date, cash, holdings)
        if EXCLUDED_SYMBOLS.intersection(set(holdings)):
            raise AssertionError('Corporate-action handler introduced an excluded holding')
        members = membership_by_date.get(execution_date)
        if members is None:
            raise ValueError(f'ไม่มี point-in-time SET50 membership สำหรับ {execution_date.date()}')
        members = set(members) - EXCLUDED_SYMBOLS
        cash, holdings, day_orders = execute_session(
            decision_date, execution_date, signals, opens, eligible, holdings, cash,
            weights, board_lot_size, set(members), classes,
        )
        if EXCLUDED_SYMBOLS.intersection({row['Symbol'] for row in day_orders}):
            raise AssertionError('Excluded security appeared in orders')
        if EXCLUDED_SYMBOLS.intersection(set(holdings)):
            raise AssertionError('Excluded security appeared in portfolio holdings')
        order_rows.extend(day_orders)
        closes = {
            symbol: (float(row.Close), execution_date)
            for symbol, row in day_rows.items()
            if row is not None and pd.notna(getattr(row, 'Close', np.nan)) and row.Close > 0
        }
        value, market_value, valuation = mark_to_market(execution_date, cash, holdings, closes, max_stale_calendar_days)
        valuation_rows.extend(valuation.to_dict('records'))
        daily_rows.append({'Date': execution_date, 'Decision_Date': decision_date, 'Cash': cash,
                           'Market_Value': market_value, 'Portfolio_Value': value,
                           'Cash_Weight': cash / value if value > 0 else np.nan, 'Holding_Count': len(holdings)})
        prediction_rows.extend(day.to_dict('records'))
    daily = pd.DataFrame(daily_rows)
    orders = pd.DataFrame(order_rows)
    valuations = pd.DataFrame(valuation_rows)
    metrics = calculate_performance_metrics(daily, initial_equity, initial_date, orders, valuations)
    return daily, orders, pd.DataFrame(prediction_rows), valuations, metrics


def run_experiment_63(run_single: Callable, **shared_inputs) -> dict:
    if not BACKTEST_GATE_OPEN:
        raise RuntimeError('BACKTEST BLOCKED: Gate ยังไม่ผ่าน; 63 combinations ไม่เริ่มทำงาน')
    if set(IMPLEMENTED_ALLOCATORS) != set(ALLOCATION_METHODS):
        raise PolicyUnresolvedError('Allocation methods ยัง implement/review ไม่ครบทั้ง 7 วิธี')
    results = {}
    for strategy_name, allocation_name in EXPERIMENT_COMBINATIONS:
        results[(strategy_name, allocation_name)] = run_single(
            strategy_name=strategy_name, allocation_name=allocation_name, **shared_inputs,
        )
    if len(results) != 63:
        raise AssertionError('จำนวน combinations ไม่ครบ 63')
    return results
#

## 13. Synthetic regression tests (mechanics only)

Fixtures ใช้ทดสอบกลไก ไม่ปะปนกับข้อมูลวิจัย และไม่ทำให้ Data Integrity Gate ผ่านแทนหลักฐานภายนอก

In [29]:
# บันทึกเฉพาะ AssertionError เป็น FAIL; exception อื่นจะถูกส่งต่อเพื่อไม่ซ่อนข้อผิดพลาด
unit_test_rows = []


def record_test(name: str, test_function: Callable) -> None:
    try:
        test_function()
    except AssertionError as exc:
        unit_test_rows.append({'Test': name, 'Status': 'FAIL', 'Detail': str(exc)})
    else:
        unit_test_rows.append({'Test': name, 'Status': 'PASS', 'Detail': 'synthetic fixture assertion passed'})


def test_t_to_next_session_uses_next_open():
    decision, next_day = pd.Timestamp('2026-01-05'), pd.Timestamp('2026-01-06')
    mapping = build_next_session_map([decision, next_day])
    assert align_decision_to_execution(decision, mapping) == next_day
    cash, holdings, orders = execute_session(decision, next_day, {'A': 'Buy'}, {'A': 101.0}, {'A': True}, {}, 1000.0,
                                              pd.Series({'A': 1.0}), 1)
    fill = next(row for row in orders if row['Status'] == 'FILLED')
    assert fill['Raw_Open'] == 101.0 and fill['Execution_Date'] == next_day and fill['Raw_Open'] != 999.0
    assert cash >= 0 and holdings['A'] > 0


def test_gap_does_not_block():
    _, _, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Buy'}, {'A': 120.0}, {'A': True}, {}, 1000.0,
                                   pd.Series({'A': 1.0}), 1)
    assert any(row['Status'] == 'FILLED' and row['Raw_Open'] == 120.0 for row in orders)


def test_slippage_and_fee_applied_once():
    buy, sell = make_fill_quote('buy', 100.0, 100), make_fill_quote('sell', 100.0, 100)
    assert math.isclose(buy.effective_price, 100.0 * (1 + SLIPPAGE_RATE), abs_tol=1e-12)
    assert math.isclose(sell.effective_price, 100.0 * (1 - SLIPPAGE_RATE), abs_tol=1e-12)
    assert math.isclose(buy.transaction_cost, buy.gross_notional * BUY_COST_RATE, abs_tol=1e-12)
    assert math.isclose(sell.transaction_cost, sell.gross_notional * SELL_COST_RATE, abs_tol=1e-12)
    assert math.isclose(buy.slippage_cost, abs(buy.effective_price - buy.raw_open) * buy.shares, abs_tol=1e-12)


def test_sell_first_unfilled_sale_does_not_fund_buy():
    cash, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Sell', 'B': 'Buy'},
                                             {'A': np.nan, 'B': 10.0}, {'A': True, 'B': True}, {'A': 100}, 0.0,
                                             pd.Series({'B': 1.0}), 1)
    assert orders[0]['Symbol'] == 'A' and orders[0]['Status'] == 'NO_FILL'
    assert any(row['Symbol'] == 'B' and row['Status'] == 'NO_FILL' for row in orders)
    assert cash == 0.0 and holdings == {'A': 100}


def test_hold_quantity_unchanged():
    cash, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Hold'}, {'A': 10.0}, {'A': True},
                                             {'A': 100}, 500.0, pd.Series(dtype=float), 1)
    assert holdings == {'A': 100} and cash == 500.0
    assert any(row['Action'] == 'Hold' and row['Status'] == 'NO_ACTION' for row in orders)


def test_buy_does_not_liquidate_hold_asset():
    cash, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Hold', 'B': 'Buy'},
                                             {'A': 10.0, 'B': 10.0}, {'A': True, 'B': True}, {'A': 100}, 0.0,
                                             pd.Series({'B': 1.0}), 1)
    assert holdings == {'A': 100} and cash == 0.0
    assert not any(row['Symbol'] == 'A' and row['Action'] == 'Sell' for row in orders)
    assert any(row['Symbol'] == 'B' and row['Status'] == 'NO_FILL' for row in orders)


def test_cash_share_invariants_and_25pct_cap():
    cash, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Buy'}, {'A': 1.0}, {'A': True}, {}, 1000.0,
                                             pd.Series({'A': 1.0}), 100)
    fill = next(row for row in orders if row['Status'] == 'FILLED')
    capped = apply_weight_cap(pd.Series({'A': 1.0}))
    mpt_cash_weights = apply_weight_cap(pd.Series({'A': 0.2}))
    assert cash >= 0 and all(shares >= 0 for shares in holdings.values())
    assert fill['Shares'] > 0 and fill['Shares'] % 100 == 0
    assert fill['Gross_Notional'] + fill['Fee'] <= 250.0 + 1e-8
    assert fill['Pre_Cost_Shares'] >= fill['Shares']
    assert fill['Pre_Cost_Notional'] >= fill['Raw_Notional']
    assert capped['A'] <= 0.25 + 1e-12 and capped.sum() <= 1.0 + 1e-12
    assert math.isclose(mpt_cash_weights['A'], 0.2, abs_tol=1e-12)
    assert math.isclose(mpt_cash_weights.sum(), 0.2, abs_tol=1e-12)


def test_missing_or_untradable_open_is_not_filled():
    _, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Buy', 'B': 'Buy'},
                                          {'A': np.nan, 'B': 10.0}, {'A': True, 'B': False}, {}, 1000.0,
                                          pd.Series({'A': 0.5, 'B': 0.5}), 1)
    assert not holdings and len([row for row in orders if row['Status'] == 'NO_FILL']) == 2
    assert all(row['No_Fill_Reason'] in {'missing_or_invalid_open', 'not_tradable_or_missing_open'} for row in orders)


def test_raw_close_and_stale_policy():
    date = pd.Timestamp('2026-02-03')
    value, market, records = mark_to_market(date, 100.0, {'A': 10}, {'A': (9.5, date)}, None)
    assert value == 195.0 and market == 95.0 and records.iloc[0]['Valuation_Price'] == 9.5
    stale_date = date - pd.Timedelta(days=1)
    stale_value, _, stale_records = mark_to_market(date, 100.0, {'A': 10}, {'A': (9.5, stale_date)}, 2)
    assert stale_value == 195.0 and bool(stale_records.iloc[0]['Is_Stale'])
    try:
        mark_to_market(date, 100.0, {'A': 10}, {'A': (9.5, stale_date)}, None)
    except ValueError as exc:
        assert 'Stale valuation' in str(exc)
    else:
        raise AssertionError('stale quote ต้องหยุดเมื่อไม่มี approved policy')
    try:
        mark_to_market(date, 100.0, {'A': 10}, {}, None)
    except ValueError as exc:
        assert 'Missing Raw Close' in str(exc)
    else:
        raise AssertionError('Missing Raw Close ต้องหยุดการประเมินพอร์ต')


def test_forced_set50_removal_exit():
    cash, holdings, orders = execute_session('2026-02-02', '2026-02-03', {'A': 'Hold'}, {'A': 10.0}, {'A': True},
                                             {'A': 100}, 0.0, pd.Series(dtype=float), 1, set50_members=set())
    assert 'A' not in holdings
    assert any(row['Reason'] == 'forced_set50_exit' and row['Status'] == 'FILLED' for row in orders)
    assert cash > 0


def test_point_in_time_membership_events():
    day1, day2, day3 = pd.to_datetime(['2026-01-01', '2026-07-01', '2027-01-01'])
    events = pd.DataFrame([
        {'Symbol': 'A', 'effective_date': day1, 'action': 'ADD'},
        {'Symbol': 'B', 'effective_date': day2, 'action': 'ADD'},
        {'Symbol': 'A', 'effective_date': day3, 'action': 'REMOVE'},
    ])
    mapping = build_membership_by_date(events, [day1, day2, day3])
    assert mapping[day1] == {'A'} and mapping[day2] == {'A', 'B'} and mapping[day3] == {'B'}


def test_unverified_corporate_action_is_rejected():
    try:
        apply_verified_corporate_actions('2026-02-03', 100.0, {'A': 100}, corporate_action_ledger)
    except (UnverifiedCorporateActionError, PolicyUnresolvedError):
        return
    raise AssertionError('unverified Corporate Action ledger ต้องถูกปฏิเสธ')


def test_prediction_uses_feature_columns_only():
    class SpyModel:
        classes_ = np.array([0, 1, 2])
        def predict(self, frame):
            self.columns_seen = list(frame.columns)
            return np.zeros(len(frame), dtype=int)
        def predict_proba(self, frame):
            return np.tile(np.array([[0.7, 0.2, 0.1]]), (len(frame), 1))
    model = SpyModel()
    row = {'Date': pd.Timestamp('2026-02-02'), 'Symbol': 'A', TARGET_COLUMN: 1, 'Actual_Target': 1}
    row.update({name: 0.1 for name in FEATURE_COLUMNS})
    result = generate_predictions(pd.DataFrame([row]), model)
    assert model.columns_seen == FEATURE_COLUMNS
    assert TARGET_COLUMN not in model.columns_seen and 'Actual_Target' not in model.columns_seen
    assert 'Actual_Target' not in result.columns and result.iloc[0]['Predicted_Class'] == 0


def test_future_rows_cannot_change_date_t_prediction_input():
    class FeatureOnlyModel:
        classes_ = np.array([0, 1, 2])
        def predict(self, frame):
            return np.where(frame['daily_return'].to_numpy() > 0, 1, 0)
        def predict_proba(self, frame):
            return np.tile(np.array([[0.4, 0.4, 0.2]]), (len(frame), 1))
    date_t = pd.Timestamp('2026-02-02')
    row = {'Date': date_t, 'Symbol': 'A', TARGET_COLUMN: 2}
    row.update({name: 0.1 for name in FEATURE_COLUMNS})
    future_a = {'Date': date_t + pd.Timedelta(days=1), 'Symbol': 'A', TARGET_COLUMN: 0, **{name: 100.0 for name in FEATURE_COLUMNS}}
    future_b = dict(future_a)
    future_b['daily_return'] = -100.0
    model = FeatureOnlyModel()
    first = generate_predictions(pd.DataFrame([row]), model)
    second = generate_predictions(pd.DataFrame([row]), model)
    assert first['Predicted_Class'].tolist() == second['Predicted_Class'].tolist()
    for future_row in [future_a, future_b]:
        try:
            generate_predictions(pd.DataFrame([row, future_row]), model)
        except ValueError as exc:
            assert 'decision date เดียว' in str(exc)
        else:
            raise AssertionError('Prediction ต้องปฏิเสธ input ที่มีวันอนาคตปะปน')


def test_excluded_symbols_are_removed_before_model_inference():
    class SpyModel:
        classes_ = np.array([0, 1, 2])
        def predict(self, frame):
            self.predict_rows = len(frame)
            return np.zeros(len(frame), dtype=int)
        def predict_proba(self, frame):
            self.probability_rows = len(frame)
            return np.tile(np.array([[0.7, 0.2, 0.1]]), (len(frame), 1))
    model = SpyModel()
    rows = []
    for symbol in ['BANPU', 'AOT']:
        row = {'Date': pd.Timestamp('2026-02-02'), 'Symbol': symbol}
        row.update({name: 0.1 for name in FEATURE_COLUMNS})
        rows.append(row)
    result = generate_predictions(pd.DataFrame(rows), model)
    assert model.predict_rows == model.probability_rows == 1
    assert result['Symbol'].tolist() == ['AOT']


def test_initial_equity_and_drawdown_baseline():
    curve = pd.DataFrame({'Date': pd.to_datetime(['2026-01-06', '2026-01-07', '2026-01-08']),
                          'Portfolio_Value': [1_100_000.0, 900_000.0, 1_200_000.0]})
    metrics = calculate_performance_metrics(curve, 1_000_000.0, pd.Timestamp('2026-01-05'))
    assert math.isclose(metrics['Total_Return'], 0.2, abs_tol=1e-12)
    assert math.isclose(metrics['Max_Drawdown'], 900_000 / 1_100_000 - 1, abs_tol=1e-12)


def test_cash_and_market_value_reconcile_after_buy_costs():
    day = pd.Timestamp('2026-02-03')
    cash, holdings, orders = execute_session(day - pd.Timedelta(days=1), day, {'A': 'Buy'}, {'A': 1.0}, {'A': True},
                                             {}, 1000.0, pd.Series({'A': 1.0}), 1)
    equity, _, _ = mark_to_market(day, cash, holdings, {'A': (1.0, day)}, None)
    fill = next(row for row in orders if row['Status'] == 'FILLED')
    expected_cost = fill['Fee'] + fill['Slippage_Cost']
    assert math.isclose(equity, 1000.0 - expected_cost, abs_tol=1e-8)


def test_approved_exclusion_applies_before_decisions():
    original = pd.DataFrame({'Symbol': ['BANPU', 'GLOBAL', 'AOT', 'BANPU']})
    filtered = apply_approved_exclusions(original, 'Synthetic fixture')
    assert filtered['Symbol'].tolist() == ['AOT']
    assert original['Symbol'].tolist() == ['BANPU', 'GLOBAL', 'AOT', 'BANPU']
    assert not filtered['Symbol'].isin(EXCLUDED_SYMBOLS).any()
    decision_date = pd.Timestamp('2026-01-05')
    next_date = pd.Timestamp('2026-01-06')
    predictions = pd.DataFrame({
        'Date': [decision_date] * 3,
        'Symbol': ['BANPU', 'GLOBAL', 'AOT'],
        'Predicted_Class': [1, 0, 0],
        'Prediction_Label': ['FOMO', 'Panic', 'Panic'],
    })
    clean_predictions = apply_approved_exclusions(predictions, 'Synthetic predictions')
    signal_rows = generate_signals(clean_predictions, 'S1', STRATEGY_MAP)
    signals = signal_rows.set_index('Symbol')['Signal'].to_dict()
    allocation_inputs = [symbol for symbol, signal in signals.items() if signal == 'Buy']
    assert allocation_inputs == ['AOT']
    cash, holdings, orders = execute_session(
        decision_date, next_date, signals, {'AOT': 10.0}, {'AOT': True}, {}, 10000.0,
        pd.Series({'AOT': 0.25}), 1, {'AOT'},
        signal_rows.set_index('Symbol')['Prediction_Label'].to_dict(),
    )
    assert not EXCLUDED_SYMBOLS.intersection(allocation_inputs)
    assert not EXCLUDED_SYMBOLS.intersection({row['Symbol'] for row in orders})
    assert not EXCLUDED_SYMBOLS.intersection(set(holdings))
    assert not EXCLUDED_SYMBOLS.intersection({row['Symbol'] for row in orders if row['Action'] == 'Sell'})


for test_name, test_function in [
    ('approved_exclusion_applies_before_decisions', test_approved_exclusion_applies_before_decisions),
    ('t_to_t_plus_1_uses_next_open', test_t_to_next_session_uses_next_open),
    ('gap_does_not_block_execution', test_gap_does_not_block),
    ('slippage_and_fee_applied_once', test_slippage_and_fee_applied_once),
    ('sell_first_unfilled_sell_no_cash', test_sell_first_unfilled_sale_does_not_fund_buy),
    ('hold_quantity_unchanged', test_hold_quantity_unchanged),
    ('buy_does_not_sell_hold_asset', test_buy_does_not_liquidate_hold_asset),
    ('cash_share_invariants_and_25pct_cap', test_cash_share_invariants_and_25pct_cap),
    ('missing_or_untradable_open_no_fill', test_missing_or_untradable_open_is_not_filled),
    ('raw_close_and_stale_policy', test_raw_close_and_stale_policy),
    ('forced_set50_removal_exit', test_forced_set50_removal_exit),
    ('point_in_time_membership_events', test_point_in_time_membership_events),
    ('unverified_corporate_action_rejected', test_unverified_corporate_action_is_rejected),
    ('prediction_uses_features_only', test_prediction_uses_feature_columns_only),
    ('excluded_symbols_removed_before_model_inference', test_excluded_symbols_are_removed_before_model_inference),
    ('no_future_row_in_date_t_prediction', test_future_rows_cannot_change_date_t_prediction_input),
    ('initial_equity_and_drawdown', test_initial_equity_and_drawdown_baseline),
    ('cash_market_value_reconciliation', test_cash_and_market_value_reconcile_after_buy_costs),
]:
    record_test(test_name, test_function)

unit_test_report = pd.DataFrame(unit_test_rows)
UNIT_TESTS_PASSED = bool(not unit_test_report.empty and unit_test_report['Status'].eq('PASS').all())
print(unit_test_report.to_string(index=False))
print('Synthetic regression tests:', 'PASS' if UNIT_TESTS_PASSED else 'FAIL')
#

                                       Test Status                             Detail
approved_exclusion_applies_before_decisions   PASS synthetic fixture assertion passed
               t_to_t_plus_1_uses_next_open   PASS synthetic fixture assertion passed
               gap_does_not_block_execution   PASS synthetic fixture assertion passed
              slippage_and_fee_applied_once   PASS synthetic fixture assertion passed
           sell_first_unfilled_sell_no_cash   PASS synthetic fixture assertion passed
                    hold_quantity_unchanged   PASS synthetic fixture assertion passed
               buy_does_not_sell_hold_asset   PASS synthetic fixture assertion passed
        cash_share_invariants_and_25pct_cap   PASS synthetic fixture assertion passed
         missing_or_untradable_open_no_fill   PASS synthetic fixture assertion passed
                 raw_close_and_stale_policy   PASS synthetic fixture assertion passed
                  forced_set50_removal_exit   PASS syn

## 14. Evidence-driven Gate Summary (fail-closed)

แต่ละ requirement มี status, evidence path และ audit detail; ขาดหลักฐานให้ BLOCKED, พบความขัดแย้งให้ FAIL และไม่มี cell ให้เปิด Gate ด้วยมือ

In [25]:
feature_notebook_path = PROJECT_ROOT / '03_feature_engineering.ipynb'
feature_notebook = json.loads(feature_notebook_path.read_text(encoding='utf-8')) if feature_notebook_path.is_file() else {}
feature_code_cells = [
    ''.join(cell.get('source', []))
    for cell in feature_notebook.get('cells', [])
    if cell.get('cell_type') == 'code'
]
feature_builder_text = next(
    (text for text in feature_code_cells if 'downloaded = yf.download(' in text),
    '',
)
feature_notebook_source = '\n'.join(feature_code_cells)
feature_download_evidence = bool(
    feature_builder_text and 'auto_adjust=True' in feature_builder_text
)
feature_download_actions_explicit = 'actions=' in feature_builder_text
feature_target_uses_next_return = bool(
    'daily_return"].shift(-1)' in feature_notebook_source
    or "daily_return'].shift(-1)" in feature_notebook_source
    or 'next_return = frame["daily_return"].shift(-1)' in feature_notebook_source
)
# สร้าง timestamped audit directory ใหม่ ไม่เขียนทับผลเดิม
run_timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = BACKTEST_DIR / f'research_audit_{run_timestamp}_{uuid.uuid4().hex[:8]}'
RUN_DIR.mkdir(parents=True, exist_ok=False)

unit_test_report_path = RUN_DIR / 'unit_test_report.csv'
unit_test_report.to_csv(unit_test_report_path, index=False)

source_paths = {
    'feature_master': feature_path, 'execution_candidate': EXECUTION_CANDIDATE_PATH,
    'execution_verified_dataset': VERIFIED_EXECUTION_PATH, 'execution_sidecar': EXECUTION_SIDECAR_PATH,
    'execution_lineage': LINEAGE_PATH, 'execution_verified_status': VERIFIED_EXECUTION_STATUS_PATH,
    'membership': MEMBERSHIP_PATH, 'membership_effective_dates': DECLARATION_PATH,
    'feature_notebook': PROJECT_ROOT / '03_feature_engineering.ipynb',
    'approved_exclusion_policy': EXCLUSION_POLICY_PATH,
    'model_notebook': PROJECT_ROOT / '04_model.ipynb', 'frozen_model': MODEL_ARTIFACT_PATH,
    'model_metadata': MODEL_METADATA_PATH, 'execution_price_audit': EXECUTION_AUDIT_PATH,
    'immutable_execution_copy': IMMUTABLE_COPY_PATH,
    'verified_execution_manifest': VERIFIED_EXECUTION_MANIFEST_PATH,
    'global_recent_SET_crosscheck': GLOBAL_RECENT_CROSSCHECK_PATH,
    'security_identity_mapping': SECURITY_IDENTITY_PATH,
    'execution_tick_audit': EXECUTION_TICK_AUDIT_PATH,
    'execution_tick_original_audit': EXECUTION_TICK_ORIGINAL_PATH,
    'execution_missing_row_reconciliation': safe_project_path(verified_execution_manifest.get('missing_rows_reconciliation_path')),
    'corporate_action_ledger': CORPORATE_ACTION_LEDGER_PATH,
    'vendor_action_inventory': CORPORATE_ACTION_VENDOR_PATH, 'banpu_audit': BANPU_AUDIT_PATH,
    'global_tick_audit': GLOBAL_TICK_AUDIT_PATH, 'previous_gate_checklist': GATE_CHECKLIST_PATH,
    'notebook_05_read_only': PROJECT_ROOT / '05_backtest.ipynb',
    'phase_a_universe_helper': PROJECT_ROOT / 'phase_a_universe.py',
}
source_inventory_rows = []
for name, path in source_paths.items():
    exists = bool(path is not None and path.is_file())
    source_inventory_rows.append({
        'Name': name, 'Path': str(path) if path is not None else None, 'Exists': exists,
        'Size_Bytes': path.stat().st_size if exists else None,
        'Modified_UTC': datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat() if exists else None,
        'SHA256': sha256_file(path) if exists else None,
    })
source_inventory = pd.DataFrame(source_inventory_rows)
source_inventory_path = RUN_DIR / 'source_inventory.csv'
source_inventory.to_csv(source_inventory_path, index=False)
feature_impact_path = RUN_DIR / 'feature_impact_audit.csv'
feature_impact_audit.to_csv(feature_impact_path, index=False)

REQUIRED_GATE_REQUIREMENTS = [
    'snapshot_identity_and_immutability', 'acquisition_manifest_complete', 'execution_source_authorized',
    'execution_missing_rows_reconciled', 'corporate_actions_reconciled_and_accounted',
    'global_identity_conversion_and_tick_verified', 'banpu_identity_prices_and_features_verified',
    'tick_schedule_verified', 'feature_lineage_and_price_basis_verified', 'frozen_model_artifact_verified',
    'no_lookahead_verified', 'execution_engine_regression_tests_passed',
    'cvar_formulation_approved', 'board_lot_and_fallback_policies_resolved',
    'point_in_time_membership_verified',
]


def status_for(verified: bool, failure_status: str = 'BLOCKED') -> str:
    return 'PASS' if bool(verified) else failure_status


snapshot_hash = sha256_file(EXECUTION_CANDIDATE_PATH) if EXECUTION_CANDIDATE_PATH and EXECUTION_CANDIDATE_PATH.is_file() else None
immutable_copy_hash = sha256_file(IMMUTABLE_COPY_PATH) if IMMUTABLE_COPY_PATH and IMMUTABLE_COPY_PATH.is_file() else None
snapshot_checklist_pass = bool(
    not existing_gate_checklist.empty
    and existing_gate_checklist.loc[
        existing_gate_checklist['Check'].astype(str).eq('Frozen snapshot hash and immutable copy'), 'Status'
    ].astype(str).str.upper().eq('PASS').any()
)
snapshot_matches = bool(snapshot_hash == EXPECTED_EXECUTION_SHA256
                        and immutable_copy_hash == EXPECTED_EXECUTION_SHA256
                        and original_snapshot_info.get('sha256') == EXPECTED_EXECUTION_SHA256
                        and lineage_data.get('original_snapshot_modified') is False
                        and snapshot_checklist_pass)
request_fields = [original_snapshot_info.get('request_interval'), original_snapshot_info.get('request_group_by'),
                  original_snapshot_info.get('requested_ticker_list')]
request_manifest_complete = all(value not in (None, '', 'UNKNOWN', 'unknown') for value in request_fields)
verified_evidence_paths = [
    safe_project_path(value)
    for value in verified_execution_manifest.get('official_evidence_paths', [])
]
verified_evidence_paths = [path for path in verified_evidence_paths if path is not None and path.is_file()]
verified_manifest_hash_matches = bool(
    VERIFIED_EXECUTION_PATH and VERIFIED_EXECUTION_PATH.is_file()
    and verified_execution_manifest.get('sha256') == sha256_file(VERIFIED_EXECUTION_PATH)
)
verified_source_available = bool(
    VERIFIED_EXECUTION_PATH and VERIFIED_EXECUTION_PATH.is_file()
    and VERIFIED_EXECUTION_MANIFEST_PATH and VERIFIED_EXECUTION_MANIFEST_PATH.is_file()
    and verified_manifest_hash_matches and verified_evidence_paths
    and verified_execution_manifest.get('price_basis') == 'raw'
    and str(verified_execution_manifest.get('source_type', '')).lower() in {'set', 'authorized'}
)
missing_reconciliation_path = safe_project_path(verified_execution_manifest.get('missing_rows_reconciliation_path'))
missing_reconciliation = read_csv_if_present(missing_reconciliation_path) if missing_reconciliation_path else pd.DataFrame()
verified_missing_counts = {
    column: int(verified_execution_df[column].isna().sum())
    for column in ['Open', 'High', 'Low', 'Close'] if column in verified_execution_df
}
row_reconciliation_pass = bool(
    not missing_reconciliation.empty
    and {'Date', 'Symbol', 'Field', 'Status', 'Evidence'}.issubset(missing_reconciliation.columns)
    and missing_reconciliation['Status'].astype(str).str.upper().eq('RECONCILED').all()
    and missing_reconciliation['Evidence'].astype('string').fillna('').str.len().gt(0).all()
    and pd.to_datetime(missing_reconciliation['Date'], errors='coerce').notna().all()
)
candidate_missing_issue_keys = set()
if not execution_candidate_df.empty and {'Date', 'Symbol', 'Open', 'High', 'Low', 'Close'}.issubset(execution_candidate_df.columns):
    candidate_dates = pd.to_datetime(execution_candidate_df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
    for field in ['Open', 'High', 'Low', 'Close']:
        for row_index in execution_candidate_df.index[execution_candidate_df[field].isna()]:
            candidate_missing_issue_keys.add((
                candidate_dates.loc[row_index],
                str(execution_candidate_df.at[row_index, 'Symbol']),
                field.upper(),
            ))
reported_missing_issue_keys = set()
if row_reconciliation_pass:
    report_dates = pd.to_datetime(missing_reconciliation['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
    reported_missing_issue_keys = set(zip(
        report_dates,
        missing_reconciliation['Symbol'].astype(str),
        missing_reconciliation['Field'].astype(str).str.upper(),
    ))
reconciliation_hash_matches = bool(
    missing_reconciliation_path and missing_reconciliation_path.is_file()
    and verified_execution_manifest.get('missing_rows_reconciliation_sha256') == sha256_file(missing_reconciliation_path)
)
row_reconciliation_pass = bool(
    row_reconciliation_pass and reconciliation_hash_matches
    and candidate_missing_issue_keys.issubset(reported_missing_issue_keys)
)
missing_rows_reconciled = bool(
    verified_source_available and verified_execution_health.get('missing_columns') == []
    and verified_execution_health.get('duplicate_keys') == 0
    and verified_missing_counts and sum(verified_missing_counts.values()) == 0
    and verified_invalid_ohlc == 0
    and row_reconciliation_pass
)
official_source_values = corporate_action_ledger.get(
    'Official_Source', pd.Series(index=corporate_action_ledger.index, dtype='string')
).astype('string').fillna('').str.strip()
official_action_sources = int(official_source_values.str.startswith('https://', na=False).sum())
applied_action_values = corporate_action_ledger.get(
    'Applied_By_Portfolio_Engine', pd.Series(False, index=corporate_action_ledger.index)
).astype('string').str.strip().str.lower()
actions_applied_count = int(applied_action_values.isin({'true', '1'}).sum())
ledger_status_values = corporate_action_ledger.get(
    'Reconciliation_Status', pd.Series(index=corporate_action_ledger.index, dtype='string')
).astype('string').fillna('UNKNOWN').str.strip().str.upper()
ledger_status_counts = ledger_status_values.value_counts().to_dict()
ledger_all_reconciled = bool(not corporate_action_ledger.empty and ledger_status_values.eq('RECONCILED').all())
ca_reconciled = bool(not corporate_action_ledger.empty
                     and official_action_sources == len(corporate_action_ledger)
                     and actions_applied_count == len(corporate_action_ledger)
                     and ledger_all_reconciled
                     and {'split_accounting_preserves_equity', 'dividend_entitlement_no_double_count',
                          'merger_conversion_accounting'}.issubset(
                              set(unit_test_report.loc[unit_test_report['Status'].eq('PASS'), 'Test'])
                          ))
def audit_truthy_column(frame: pd.DataFrame, column: str) -> pd.Series:
    values = frame.get(column, pd.Series(False, index=frame.index))
    return values.astype('string').str.strip().str.lower().isin({'true', '1'})


global_summary = {
    'rows': int(len(global_tick_audit)),
    'strict_tick_aligned': int(audit_truthy_column(global_tick_audit, 'Strict_Tick_Aligned').sum()),
    'diagnostic_tick_aligned': int(audit_truthy_column(global_tick_audit, 'Diagnostic_Tick_Aligned').sum()),
    'official_action_confirmed': bool(
        not global_tick_audit.empty
        and 'Official_Action_Confirmed' in global_tick_audit.columns
        and audit_truthy_column(global_tick_audit, 'Official_Action_Confirmed').all()
    ),
    'recent_SET_crosscheck_rows': int(len(global_recent_crosscheck)),
    'recent_SET_OHLC_matches': int(
        global_recent_crosscheck[['Open_Matches_SET', 'High_Matches_SET', 'Low_Matches_SET', 'Close_Matches_SET']]
        .apply(lambda column: column.astype('string').str.strip().str.lower().isin({'true', '1'}))
        .all(axis=1).sum()
    ) if {'Open_Matches_SET', 'High_Matches_SET', 'Low_Matches_SET', 'Close_Matches_SET'}.issubset(global_recent_crosscheck.columns) else 0,
    'classification_counts': global_tick_audit.get(
        'Anomaly_Classification', pd.Series(dtype='string')
    ).astype('string').fillna('UNKNOWN').value_counts().to_dict(),
}
banpu_trading_status = banpu_audit.get(
    'Trading_Status', pd.Series('', index=banpu_audit.index, dtype='string')
).astype('string').fillna('')
banpu_summary = {
    'audit_rows': int(len(banpu_audit)),
    'missing_open': int(audit_truthy_column(banpu_audit, 'Missing_Open').sum()),
    'missing_close': int(audit_truthy_column(banpu_audit, 'Missing_Close').sum()),
    'zero_volume_or_suspended': int((
        audit_truthy_column(banpu_audit, 'Zero_Volume_Bar')
        | banpu_trading_status.str.contains('SUSPEND', case=False, regex=False)
    ).sum()),
    'trading_eligible_true': int(audit_truthy_column(banpu_audit, 'Trading_Eligible').sum()),
}
security_identity_status_counts = security_identity_mapping.get(
    'Evidence_Status', pd.Series(dtype='string')
).astype('string').fillna('UNKNOWN').value_counts().to_dict()
security_identity_verified = bool(
    not security_identity_mapping.empty
    and 'Evidence_Status' in security_identity_mapping.columns
    and security_identity_mapping['Evidence_Status'].astype('string').str.upper().eq('PASS').all()
)
global_required_columns = {
    'Strict_Tick_Aligned', 'Official_Action_Confirmed', 'Research_Execution_Eligible',
}
global_verified = bool(not global_tick_audit.empty
                       and global_required_columns.issubset(global_tick_audit.columns)
                       and global_summary['official_action_confirmed']
                       and global_summary['recent_SET_crosscheck_rows'] > 0
                       and global_summary['strict_tick_aligned'] == global_summary['rows']
                       and audit_truthy_column(global_tick_audit, 'Research_Execution_Eligible').all())
banpu_required_columns = {'Missing_Open', 'Missing_Close', 'Trading_Eligible'}
banpu_verified = bool(not banpu_audit.empty and banpu_summary['missing_open'] == 0
                      and banpu_summary['missing_close'] == 0
                      and banpu_summary['trading_eligible_true'] == len(banpu_audit)
                      and banpu_required_columns.issubset(banpu_audit.columns)
                      and security_identity_verified)
tick_schedule_candidates = [
    path for base in [DATASET_DIR, BACKTEST_DIR] if base.is_dir()
    for path in base.rglob('*.csv')
    if 'tick' in path.name.lower() and path not in {GLOBAL_TICK_AUDIT_PATH, EXECUTION_TICK_AUDIT_PATH, EXECUTION_TICK_ORIGINAL_PATH}
]
tick_schedule_verified = any(
    {'effective_date', 'source_url'}.issubset(set(pd.read_csv(path, nrows=0).columns))
    and pd.read_csv(path)['source_url'].astype('string').fillna('').str.startswith('https://www.set.or.th/').all()
    for path in tick_schedule_candidates
)
feature_lineage_verified = bool(feature_download_evidence and feature_path.is_file()
                                and feature_builder_text and model_saved_from_selected_pipeline
                                and (not EXECUTION_CANDIDATE_PATH or feature_path.resolve() != EXECUTION_CANDIDATE_PATH.resolve()))
no_lookahead_verified = bool(MODEL_ARTIFACT_VERIFIED and TARGET_COLUMN not in FEATURE_COLUMNS
                             and feature_target_uses_next_return and validated_session_calendar is not None)
methodology_resolved = bool(CVAR_FORMULATION and COVARIANCE_ESTIMATOR and ALLOCATOR_FALLBACK_POLICY)
policies_resolved = bool(BOARD_LOT_SIZE is not None and MAX_STALE_CALENDAR_DAYS is not None
                         and COVARIANCE_ESTIMATOR is not None and ALLOCATOR_FALLBACK_POLICY is not None
                         and set(IMPLEMENTED_ALLOCATORS) == set(ALLOCATION_METHODS))
membership_dates_verified = bool(
    membership_source_urls_official and not membership_events.empty
    and len(observed_membership_by_date) == len(observed_feature_dates)
)

gate_rows = [
    (REQUIRED_GATE_REQUIREMENTS[0], status_for(snapshot_matches, 'FAIL'), LINEAGE_PATH,
     f'current_sha256={snapshot_hash}; immutable_copy={IMMUTABLE_COPY_PATH}; copy_sha256={immutable_copy_hash}; pinned_sha256={EXPECTED_EXECUTION_SHA256}; checklist_pass={snapshot_checklist_pass}'),
    (REQUIRED_GATE_REQUIREMENTS[1], status_for(request_manifest_complete), EXECUTION_SIDECAR_PATH,
     f'interval={request_fields[0]}; group_by={request_fields[1]}; requested_ticker_list={request_fields[2]}'),
    (REQUIRED_GATE_REQUIREMENTS[2], status_for(verified_source_available), VERIFIED_EXECUTION_STATUS_PATH,
     f'verified_file_exists={bool(VERIFIED_EXECUTION_PATH and VERIFIED_EXECUTION_PATH.is_file())}; manifest_exists={bool(VERIFIED_EXECUTION_MANIFEST_PATH and VERIFIED_EXECUTION_MANIFEST_PATH.is_file())}; manifest_hash_matches={verified_manifest_hash_matches}; official_evidence_files={len(verified_evidence_paths)}; vendor={lineage_data.get("source")}'),
    (REQUIRED_GATE_REQUIREMENTS[3], status_for(missing_rows_reconciled), missing_reconciliation_path or EXECUTION_AUDIT_PATH,
     f'candidate_missing_OHLC={candidate_ohlc_missing}; verified_missing_OHLC={verified_missing_counts}; verified_invalid_OHLC_rows={verified_invalid_ohlc}; missing_issues={len(candidate_missing_issue_keys)}; reported_issues={len(reported_missing_issue_keys)}; reconciliation_hash_matches={reconciliation_hash_matches}; reconciliation_rows_pass={row_reconciliation_pass}'),
    (REQUIRED_GATE_REQUIREMENTS[4], status_for(ca_reconciled), CORPORATE_ACTION_LEDGER_PATH,
     f'ledger_rows={len(corporate_action_ledger)}; official_urls={official_action_sources}; applied_by_engine={actions_applied_count}; status_counts={ledger_status_counts}'),
    (REQUIRED_GATE_REQUIREMENTS[5], status_for(global_verified), GLOBAL_TICK_AUDIT_PATH,
     f'anomalies={global_summary["rows"]}; strict_tick_aligned={global_summary["strict_tick_aligned"]}; diagnostic_aligned={global_summary["diagnostic_tick_aligned"]}; official_confirmed={global_summary["official_action_confirmed"]}; SET_recent_rows={global_summary["recent_SET_crosscheck_rows"]}; OHLC_matches={global_summary["recent_SET_OHLC_matches"]}; classifications={global_summary["classification_counts"]}'),
    (REQUIRED_GATE_REQUIREMENTS[6], status_for(banpu_verified), BANPU_AUDIT_PATH,
     f'audit_rows={banpu_summary["audit_rows"]}; missing_open={banpu_summary["missing_open"]}; missing_close={banpu_summary["missing_close"]}; zero_volume_or_suspended={banpu_summary["zero_volume_or_suspended"]}; identity_status={security_identity_status_counts}; official identity/calendar unresolved'),
    (REQUIRED_GATE_REQUIREMENTS[7], status_for(tick_schedule_verified), GLOBAL_TICK_AUDIT_PATH,
     f'schedule_candidates={[str(path) for path in tick_schedule_candidates]}; no date-effective official tick schedule or exchange-verified price basis'),
    (REQUIRED_GATE_REQUIREMENTS[8], status_for(feature_lineage_verified), PROJECT_ROOT / '03_feature_engineering.ipynb',
     f'feature_auto_adjust_true={bool(feature_download_evidence)}; actions_parameter_explicit={feature_download_actions_explicit}; execution_auto_adjust={lineage_data.get("auto_adjust")}; feature and execution snapshots are separate; GLOBAL/BANPU feature rebuild not verified'),
    (REQUIRED_GATE_REQUIREMENTS[9], status_for(MODEL_ARTIFACT_VERIFIED), MODEL_ARTIFACT_PATH or models_root,
     f'artifact={MODEL_ARTIFACT_PATH}; sha256={sha256_file(MODEL_ARTIFACT_PATH) if MODEL_ARTIFACT_PATH else None}; checks={model_check_details}'),
    (REQUIRED_GATE_REQUIREMENTS[10], status_for(no_lookahead_verified), PROJECT_ROOT / '06_research_backtest.ipynb',
     f'prediction selects only {FEATURE_COLUMNS}; target is a next-return label; official session calendar and full point-in-time feature lineage not verified'),
    (REQUIRED_GATE_REQUIREMENTS[11], status_for(UNIT_TESTS_PASSED, 'FAIL'), unit_test_report_path,
     f'synthetic_checks={len(unit_test_report)}; passed={int(unit_test_report["Status"].eq("PASS").sum())}; failed={int(unit_test_report["Status"].eq("FAIL").sum())}; tests do not certify data integrity'),
    (REQUIRED_GATE_REQUIREMENTS[12], status_for(bool(CVAR_FORMULATION)), PROJECT_ROOT / '06_research_backtest.ipynb',
     'CVaR objective/constraint ยังไม่ได้กำหนดและอนุมัติ; ห้ามใช้ minimize-CVaR with sum(w)<=1 ซึ่งอาจเลือก all-cash'),
    (REQUIRED_GATE_REQUIREMENTS[13], status_for(policies_resolved), PROJECT_ROOT / '06_research_backtest.ipynb',
     f'board_lot={BOARD_LOT_SIZE}; max_stale_days={MAX_STALE_CALENDAR_DAYS}; covariance={COVARIANCE_ESTIMATOR}; fallback={ALLOCATOR_FALLBACK_POLICY}; implemented_allocators={sorted(IMPLEMENTED_ALLOCATORS)}'),
    (REQUIRED_GATE_REQUIREMENTS[14], status_for(membership_dates_verified), DECLARATION_PATH,
     f'membership_events={len(membership_events)}; source_urls_present={membership_source_urls_present}; all_primary_SET_urls={membership_source_urls_official}; observed_map_is_not_official_calendar'),
]
GATE_EVIDENCE_REPORT = pd.DataFrame([
    {'requirement': requirement, 'status': status, 'evidence_path': str(path), 'audit_detail': detail}
    for requirement, status, path, detail in gate_rows
])


def derive_gate_state(report: pd.DataFrame) -> tuple[bool, list[str]]:
    required_columns = {'requirement', 'status', 'evidence_path', 'audit_detail'}
    if report.empty or not required_columns.issubset(report.columns):
        return False, ['Evidence Gate Report missing/invalid']
    if report['requirement'].duplicated().any():
        return False, ['duplicate requirements in Evidence Gate Report']
    problems = []
    for requirement in REQUIRED_GATE_REQUIREMENTS:
        rows = report.loc[report['requirement'].eq(requirement)]
        if len(rows) != 1:
            problems.append(f'{requirement}: missing or duplicate')
            continue
        row = rows.iloc[0]
        if str(row['status']).upper() != 'PASS':
            problems.append(f'{requirement}: {row["status"]}')
        if not str(row['evidence_path']).strip() or not str(row['audit_detail']).strip():
            problems.append(f'{requirement}: missing evidence path/detail')
    return len(problems) == 0, problems


BACKTEST_GATE_OPEN, GATE_BLOCKERS = derive_gate_state(GATE_EVIDENCE_REPORT)
gate_report_path = RUN_DIR / 'gate_evidence.csv'
GATE_EVIDENCE_REPORT.to_csv(gate_report_path, index=False)
print(GATE_EVIDENCE_REPORT[['requirement', 'status', 'audit_detail']].to_string(index=False, max_colwidth=180))
print('RESEARCH BACKTEST GATE:', 'OPEN' if BACKTEST_GATE_OPEN else 'CLOSED')
print('Blockers:', GATE_BLOCKERS)
#

                                 requirement  status                                                                                                                                                                         audit_detail
          snapshot_identity_and_immutability    PASS current_sha256=14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a; immutable_copy=D:\Project_Github\set_fear_index\backtest_results\execution_data_v2\execution_ohl...
               acquisition_manifest_complete BLOCKED                                                                                                                    interval=UNKNOWN; group_by=UNKNOWN; requested_ticker_list=UNKNOWN
                 execution_source_authorized BLOCKED                                verified_file_exists=False; manifest_exists=False; manifest_hash_matches=False; official_evidence_files=0; vendor=Yahoo Finance via yfinance.download
           execution_missing_rows_reconciled BLOCKED candidate_m

## 15. Outputs, provenance manifest และ run status

Gate ปิด: สร้างเฉพาะ audit evidence ใน timestamped directory ใหม่; ไม่สร้าง Strategy Performance, Transaction Log, Daily Equity หรือผล 63 combinations

In [26]:
# เก็บเฉพาะ audit evidence และ manifest; ไม่บันทึก strategy results เมื่อ Gate ปิด
def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


import importlib.util
PHASE_A_HELPER_PATH = PROJECT_ROOT / 'phase_a_universe.py'
phase_a_spec = importlib.util.spec_from_file_location('phase_a_universe', PHASE_A_HELPER_PATH)
if phase_a_spec is None or phase_a_spec.loader is None:
    raise ImportError(f'Cannot load Phase A audit helper: {PHASE_A_HELPER_PATH}')
phase_a_module = importlib.util.module_from_spec(phase_a_spec)
phase_a_spec.loader.exec_module(phase_a_module)
phase_a_context = {
    'run_dir': RUN_DIR, 'project_root': PROJECT_ROOT,
    'exclusion_policy_path': EXCLUSION_POLICY_PATH,
    'execution_candidate_path': EXECUTION_CANDIDATE_PATH,
    'execution_candidate_df': execution_candidate_df, 'feature_df': feature_df,
    'feature_path': feature_path, 'feature_columns': FEATURE_COLUMNS,
    'observed_feature_dates': observed_feature_dates,
    'membership_by_date': observed_membership_by_date,
    'build_next_session_map': build_next_session_map,
    'membership_source_urls_official': membership_source_urls_official,
    'official_calendar_available': validated_session_calendar is not None,
    'verified_execution_source': verified_source_available,
    'verified_execution_manifest': verified_execution_manifest,
    'corporate_action_ledger': corporate_action_ledger,
    'security_identity_mapping': security_identity_mapping,
    'tick_audit': execution_tick_original_audit,
    'exclusion_symbols': EXCLUDED_SYMBOLS,
    'expected_execution_sha256': EXPECTED_EXECUTION_SHA256,
    'protected_paths': {
        'execution_snapshot': EXECUTION_CANDIDATE_PATH,
        'immutable_execution_copy': IMMUTABLE_COPY_PATH,
        'master_dataset': feature_path,
        'frozen_model': MODEL_ARTIFACT_PATH,
        'membership_history': MEMBERSHIP_PATH,
        'membership_declarations': DECLARATION_PATH,
        'backtest_05': PROJECT_ROOT / '05_backtest.ipynb',
    },
    'sha256_file': sha256_file,
    'notebook_data': json.loads((PROJECT_ROOT / '06_research_backtest.ipynb').read_text(encoding='utf-8')),
    'global_summary': global_summary,
    'execution_provider': lineage_data.get('source'),
    'execution_auto_adjust': lineage_data.get('auto_adjust'),
    'feature_auto_adjust_true': feature_download_evidence,
    'feature_actions_explicit': feature_download_actions_explicit,
    'official_action_sources': official_action_sources,
    'actions_applied_count': actions_applied_count,
    'membership_path': MEMBERSHIP_PATH,
    'global_tick_path': GLOBAL_TICK_AUDIT_PATH,
    'corporate_action_path': CORPORATE_ACTION_LEDGER_PATH,
    'verified_execution_status_path': VERIFIED_EXECUTION_STATUS_PATH,
    'model_path': MODEL_ARTIFACT_PATH,
    'gate_report_path': gate_report_path,
    'backtest_gate_open': BACKTEST_GATE_OPEN,
    'unit_test_report': unit_test_report,
    'unit_test_report_path': unit_test_report_path,
}
phase_a_result = phase_a_module.write_phase_a_artifacts(phase_a_context)
phase_a_output_paths = phase_a_result['output_paths']

output_hashes = {path.name: sha256_file(path) for path in
                 [unit_test_report_path, source_inventory_path, feature_impact_path, gate_report_path, *phase_a_output_paths.values()]}
source_hashes = {row['Name']: row['SHA256'] for row in source_inventory_rows if row['SHA256'] is not None}
run_manifest = {
    'schema_version': 'research-backtest-audit-v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_directory': str(RUN_DIR),
    'run_status': 'BLOCKED' if not BACKTEST_GATE_OPEN else 'GATE_OPEN_NOT_EXECUTED',
    'backtest_gate_open': bool(BACKTEST_GATE_OPEN),
    'backtest_63_sets_executed': False,
    'strategy_result_files_created': False,
    'yfinance_redownload_performed': False,
    'original_snapshot_modified': False,
    'project_root': str(PROJECT_ROOT),
    'notebook_source_sha256': notebook_source_sha256(PROJECT_ROOT / '06_research_backtest.ipynb'),
    'model_artifact_path': str(MODEL_ARTIFACT_PATH) if MODEL_ARTIFACT_PATH else None,
    'model_artifact_sha256': sha256_file(MODEL_ARTIFACT_PATH) if MODEL_ARTIFACT_PATH else None,
    'model_metadata_path': str(MODEL_METADATA_PATH) if MODEL_METADATA_PATH else None,
    'execution_candidate_path': str(EXECUTION_CANDIDATE_PATH) if EXECUTION_CANDIDATE_PATH else None,
    'execution_candidate_sha256': snapshot_hash,
    'verified_execution_path': str(VERIFIED_EXECUTION_PATH) if VERIFIED_EXECUTION_PATH else None,
    'source_hashes': source_hashes,
    'output_hashes': output_hashes,
    'environment': {
        'python': sys.version, 'platform': platform.platform(), 'pandas': pd.__version__,
        'numpy': np.__version__, 'scikit_learn': sklearn.__version__,
        'joblib': package_version('joblib'), 'pyarrow': package_version('pyarrow'),
    },
    'configuration': {
        'features': FEATURE_COLUMNS,
        'cost_assumptions': {'buy_fee_rate': BUY_COST_RATE, 'sell_fee_rate': SELL_COST_RATE, 'slippage_rate': SLIPPAGE_RATE},
        'sharpe_daily_risk_free_rate_assumption': SHARPE_RISK_FREE_RATE_DAILY,
        'execution_lag_sessions': EXECUTION_LAG_SESSIONS, 'gap_limit': GAP_LIMIT,
        'max_buy_weight_per_stock': MAX_BUY_WEIGHT_PER_STOCK, 'board_lot_size': BOARD_LOT_SIZE,
        'max_stale_calendar_days': MAX_STALE_CALENDAR_DAYS, 'covariance_estimator': COVARIANCE_ESTIMATOR,
        'cvar_formulation': CVAR_FORMULATION, 'allocator_fallback_policy': ALLOCATOR_FALLBACK_POLICY,
    },
    'unit_test_summary': {
        'passed': int(unit_test_report['Status'].eq('PASS').sum()),
        'failed': int(unit_test_report['Status'].eq('FAIL').sum()),
        'report_path': str(unit_test_report_path),
    },
    'gate_report_path': str(gate_report_path),
    'phase_A': {
        'status': phase_a_result['overall_status'],
        'predictor_dates': phase_a_result['predictor_dates'],
        'member_date_rows': phase_a_result['member_date_rows'],
        'excluded_member_date_rows': phase_a_result['excluded_member_date_rows'],
        'tradable_rows': phase_a_result['tradable_rows'],
        'candidate_data_coverage': phase_a_result['candidate_data_coverage'],
        'candidate_eligibility_coverage': phase_a_result['candidate_eligibility_coverage'],
        'verified_execution_coverage': phase_a_result['verified_execution_coverage'],
        'protected_hashes_unchanged': phase_a_result['protected_hashes_unchanged'],
        'model_retrained': phase_a_result['model_retrained'],
        'backtest_63_sets_executed': phase_a_result['backtest_63_sets_executed'],
        'deliverables': {name: str(path) for name, path in phase_a_output_paths.items()},
    },
    'gate_blockers': GATE_BLOCKERS,
    'outputs_are_audit_only': True,
}
manifest_path = RUN_DIR / 'run_manifest.json'
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

print('Audit directory:', RUN_DIR)
print('Files written:', sorted(path.name for path in RUN_DIR.iterdir()))
print('Run status:', run_manifest['run_status'])
print('63 combinations executed:', run_manifest['backtest_63_sets_executed'])
#

Audit directory: D:\Project_Github\set_fear_index\backtest_results\research_audit_20261116T044415991651Z_ee2aa7e1
Files written: ['exclusion_audit.csv', 'execution_excluded_symbols.csv', 'feature_impact_audit.csv', 'gate_evidence.csv', 'phase_A_gate.csv', 'run_manifest.json', 'security_level_integrity_audit.csv', 'source_inventory.csv', 'tradable_universe_2026.csv', 'unit_test_report.csv', 'universe_coverage_report.md']
Run status: BLOCKED
63 combinations executed: False


## 16. Completion checklist และสถานะปัจจุบัน

- Feature source (`auto_adjust=True`) และ Execution Snapshot (`auto_adjust=False`) เป็นคนละ dataset; `actions` ของ Feature Build ไม่ได้ระบุ
- Frozen Softmax artifact ถูกตรวจจากไฟล์จริงใน repository; ไม่มีการ fit/refit
- Prediction function รับเฉพาะ Feature columns และแยก Target เป็น label สำหรับ audit เท่านั้น
- Synthetic mechanics tests ไม่ใช่หลักฐาน Data Integrity
- Gate เปิดได้เมื่อทุก requirement ใน `gate_evidence.csv` เป็น PASS พร้อม evidence; สถานะที่ตรวจพบปัจจุบันต้องเป็น CLOSED
- เมื่อ Gate ปิด จะไม่สร้างหรืออ้างผล Backtest 63 combinations
- ยังต้องมี acquisition manifest หรือ trusted reconciliation, verified execution OHLCV/official patch+calendar, official Corporate Action terms, GLOBAL identity/tick evidence, BANPU identity/calendar/prices/features, official tick schedule และอนุมัติ CVaR/covariance/board-lot/stale/fallback policy

## 17. Approved security exclusion

- BANPU มีสถานะ `APPROVED_RESEARCH_EXCLUSION` ตั้งแต่ 2026-01-01 ถึง 2026-12-31 ตาม `backtest_results/research_controls/execution_excluded_symbols.csv`.
- Exclusion ถูกใช้กับ Predictions, Execution rows, signals, allocation inputs, membership ที่ส่งเข้า engine, orders และ holdings; ไม่ลบแถวจาก Master Dataset หรือแก้ Model/Target.
- Gate ยังคงปิดจนกว่า eligible execution data, Corporate Actions และ Critical Research Integrity checks จะผ่านหลักฐานจริง; ห้ามรัน 63 combinations ระหว่างนี้.
